# 아마존 리뷰 한국어 번역 → 감성(별점) 모델 재테스트 노트북

**이 노트북이 하는 일**
1. 이전 노트북(`amazon_sentiment_finetune.ipynb`)에서 뽑은 **같은 리뷰 2만 건과 같은 학습/검증/테스트 분할**을 그대로 불러옵니다.
2. 영→한 **번역 모델과 번역 LLM을 리뷰 300건으로 먼저 비교**합니다. 이번 판은 **감성이 그대로 옮겨지는지**를 중심으로 잽니다(감성 일치율, 극성 반전율, 강도 유지율, 분포 유사도, Gemini 별점 판정).
3. 감성 유지 종합 1위 번역 모델로 **2만 건 전체를 한국어로 번역**합니다(드라이브에 중간 저장, 끊겨도 이어서 실행).
4. 한국어 번역문으로 **한국어·다국어 감성 모델을 학습 전/후로 평가**하고, 이전 **영어 원문 결과와 같은 테스트 3,000건에서 비교**합니다.
5. 항목별 정확도, 원래값/예측값 표, **비교 화면(HTML)** 을 드라이브에 저장합니다.

**전체 흐름**
`이전 샘플·분할 재현 → 번역 후보 비교(300건, 감성 유지 중심) → 번역 모델 선택 → 전체 번역 → 한국어 모델 [학습 전 평가 → 파인튜닝 → 테스트] → 영어 결과와 비교 → 대시보드`

**번역 후보 (기본으로 켜진 것)**

| 이름 | 모델 | 종류 | 라이선스 | 비고 |
|---|---|---|---|---|
| Gemini-3.5-FlashLite | Gemini API | API | 상용 API | `GEMINI_KEY`가 있을 때만 |
| Papago | 네이버 클라우드 Papago API | API | 유료 API | `PAPAGO_CLIENT_ID`, `PAPAGO_CLIENT_SECRET`이 있을 때만 |
| NLLB-600M / 1.3B | facebook/nllb-200-distilled-* | 번역 전용 | CC-BY-NC-4.0 | 지난 실행 1위(1.3B) |
| NHNDQ-NLLB-en2ko | NHNDQ/nllb-finetuned-en2ko | 번역 전용 | CC-BY-4.0 표기 | NLLB를 한국어로 추가 학습 |
| M2M100-418M / 1.2B | facebook/m2m100_* | 번역 전용 | MIT | |
| Gugugo-7B | squarelike/Gugugo-koen-7B-V1.1 | 번역 LLM | Apache-2.0 | Llama-2-ko 기반 영한 번역. 4비트 |
| Rosetta-4B | yanolja/YanoljaNEXT-Rosetta-4B | 번역 LLM | Gemma | 카드 기준 WMT24++ en→ko chrF++ 31.31 |
| EXAONE-3.5-7.8B | LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct | 범용 LLM | EXAONE NC | 영어·한국어 이중언어. 4비트 |
| Qwen2.5-7B | Qwen/Qwen2.5-7B-Instruct | 범용 LLM | Apache-2.0 | 4비트 |
| iris-7B | davidkim205/iris-7b | 번역 LLM | Apache-2.0 | Mistral 기반 한영 번역. 4비트 |
| llama3-instrucTrans-8B | nayohan/llama3-instrucTrans-enko-8b | 번역 LLM | Llama 3 | Llama-3 기반 영한 번역. 4비트 |

꺼 둔 후보(설정에서 `enabled=True`로 켜기): EXAONE-3.5-2.4B, Qwen3-4B/8B, Rosetta-12B, TowerInstruct-7B, Tower-Plus-9B, NLLB-3.3B, MADLAD-3B, OPUS-MT(지난 실행에서 56% 실패).
번역 LLM에는 "감성의 극성과 세기, 비꼼, 강조를 그대로 유지하라"는 지시문을 넣었습니다. 모델 카드에 정해진 번역 형식이 있는 모델(Gugugo, iris, llama3-instrucTrans, Rosetta)은 그 형식을 따릅니다.

**한국어 감성 모델**: KcELECTRA(한국어 댓글로 사전학습), KLUE-RoBERTa(한국어), nlptown-mBERT, tabularisai-DistilBERT, clapAI-XLMR(다국어 감성).

**사전 준비**
- Colab `런타임 > 런타임 유형 변경`에서 **GPU(T4 이상)** 를 고르세요.
- 이전 노트북을 실행해서 드라이브 `MyDrive/voc_amazon_finetune/`에 `sample_raw.parquet`와 `results/*.json`이 있어야 합니다.
- (선택) Colab Secrets에 `GEMINI_KEY`(Gemini 번역·별점 판정), `PAPAGO_CLIENT_ID`·`PAPAGO_CLIENT_SECRET`(Papago), `HF_TOKEN`(다운로드 제한 완화, 게이트 모델·CometKiwi 사용 시 필수)을 넣으세요.
- 지난 실행의 `voc_amazon_ko/translations/` 캐시가 있으면 이미 번역한 후보는 다시 번역하지 않습니다. 새로 넣은 번역 LLM만 번역합니다.
- 결과는 `MyDrive/voc_amazon_ko/`에 저장됩니다.

**꼭 알아둘 점**
- 이 실험은 "**번역된 영어 리뷰**"의 한국어 성능입니다. 한국 소비자가 직접 쓴 리뷰와는 말투가 달라서, 실제 한국 리뷰 성능은 따로 확인해야 합니다.
- 정답 한국어 번역이 없어서 BLEU 같은 참조 기반 점수는 못 씁니다. 원문과 번역문을 같은 판정기에 넣어 **감성이 얼마나 그대로 남는지**를 재고, 역번역 chrF는 직역일수록 높게 나오는 치우침이 있어 비중을 낮췄습니다.
- 이 프로젝트는 비상업용이라 비상업 라이선스(NLLB, EXAONE, Tower 등) 모델도 후보에 넣었습니다.
- 번역 LLM은 번역 전용 모델보다 훨씬 느립니다. 7B 이상은 4비트로 줄여 올리므로 T4에서도 돌지만, 후보마다 300건 번역에 시간이 꽤 걸립니다.
- 시간 예상은 하지 않았습니다. 번역 비교와 전체 번역은 GPU 상태와 모델 크기에 따라 크게 달라지므로, 진행 막대를 보고 필요하면 `MT_EVAL_N`이나 후보 수를 줄이세요.
- 이 노트북의 **모델 다운로드·번역·학습 부분은 작성 환경에서 Hugging Face와 GPU를 쓸 수 없어 실행해 보지 못했습니다.** 텍스트 처리, 평가, 표, 대시보드 코드는 가짜 데이터로 실행을 확인했습니다.

## 0. 패키지 설치와 불러오기

In [1]:
# ── 패키지 설치 ─────────────────────────────────────────────
# transformers, accelerate : 번역·감성 모델을 내려받아 실행/학습
# sentencepiece, protobuf : NLLB, M2M100, MADLAD, XLM-R 토크나이저에 필요
# sacrebleu : 역번역 chrF 점수 계산
# google-genai : Gemini API (GEMINI_KEY가 있을 때만 사용)
# bitsandbytes : 7B 이상 번역 LLM을 4비트로 줄여 T4(16GB)에 올릴 때 사용
%pip install -q transformers accelerate bitsandbytes sentencepiece protobuf sacremoses sacrebleu scikit-learn pandas pyarrow tqdm google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 11.5 MB/s eta 0:00:00


In [2]:
# ── 기본 라이브러리 ─────────────────────────────────────────
import os, re, json, time, random, gc, html, datetime as dt
import requests
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, mean_absolute_error
from IPython.display import display, HTML

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

pd.set_option('display.max_colwidth', 160)
print('Colab 여부:', IN_COLAB)

Colab 여부: True


## 1. 실험 설정
바꾸고 싶은 값은 **이 셀만** 고치면 됩니다.
샘플·분할 관련 값(`N_PER_CLASS`, `SPLIT`, `SEED`)은 이전 노트북과 **같아야** 영어 결과와 같은 테스트셋으로 비교됩니다.

In [3]:
# ===== 1-1. 폴더 =====
DRIVE_ROOT   = '/content/drive/MyDrive'
PREV_DIR_NAME = 'voc_amazon_finetune'   # 이전 노트북 결과 폴더 (sample_raw.parquet, results/*.json)
OUT_DIR_NAME  = 'voc_amazon_ko'         # 이 노트북 결과 폴더
USE_DRIVE    = True
RESUME       = True                     # 이미 끝난 번역·모델 결과가 있으면 건너뜀

# ===== 1-2. 이전 노트북과 같아야 하는 값 =====
CATEGORIES  = ['Fashion', 'Clothing·Shoes']
START_DATE, END_DATE = '2020-10-01', '2023-10-01'   # 화면 표시용 (END_DATE는 포함하지 않음)
N_PER_CLASS = 2000
SPLIT       = (0.70, 0.15, 0.15)
SEED        = 42

# ===== 1-3. 번역 후보 =====
# kind: gemini(API) / nllb / m2m / marian / madlad (번역 전용 seq2seq) / chatllm (번역 LLM)
# enabled=False인 후보는 비교에서 빠집니다. 크고 느린 모델은 기본으로 꺼 두었습니다.
MT_CANDIDATES = [
    # ── API (키가 있을 때만) ──
    dict(short='Gemini-3.5-FlashLite', kind='gemini', model='gemini-3.5-flash-lite',          license='상용 API',     enabled=True),
    dict(short='Papago',               kind='papago',                                         license='상용 API(유료)', enabled=True),
    # ── 번역 전용 모델 (문장 덩어리 단위 번역) ──
    dict(short='NLLB-600M',            kind='nllb',   hf='facebook/nllb-200-distilled-600M',  license='CC-BY-NC-4.0', enabled=True),
    dict(short='NLLB-1.3B',            kind='nllb',   hf='facebook/nllb-200-distilled-1.3B',  license='CC-BY-NC-4.0', enabled=True),
    dict(short='NHNDQ-NLLB-en2ko',     kind='nllb',   hf='NHNDQ/nllb-finetuned-en2ko',        license='CC-BY-4.0(기반 NC)', enabled=True),
    dict(short='M2M100-418M',          kind='m2m',    hf='facebook/m2m100_418M',              license='MIT',          enabled=True),
    dict(short='M2M100-1.2B',          kind='m2m',    hf='facebook/m2m100_1.2B',              license='MIT',          enabled=True),
    dict(short='OPUS-MT-big',          kind='marian', hf='Helsinki-NLP/opus-mt-tc-big-en-ko', license='CC-BY-4.0',    enabled=False),  # 지난 실행에서 56% 실패
    dict(short='NLLB-3.3B',            kind='nllb',   hf='facebook/nllb-200-3.3B',            license='CC-BY-NC-4.0', enabled=False),
    dict(short='MADLAD-3B',            kind='madlad', hf='google/madlad400-3b-mt',            license='Apache-2.0',   enabled=False, fp32=True),
    # ── 번역 LLM (리뷰 통째로 번역). quant='4bit'는 7B 이상을 T4에 올리기 위한 설정 ──
    # prompt='chat'이면 채팅 템플릿 사용. system_prompt를 안 주면 감성 유지를 강조한 LLM_INSTRUCTION을 씀.
    # 모델 카드에 정해진 번역 형식이 있으면 그대로 따름 (Gugugo, iris, nayohan, Rosetta, Tower).
    dict(short='Gugugo-7B',            kind='llm', hf='squarelike/Gugugo-koen-7B-V1.1',       license='Apache-2.0',   enabled=True, quant='4bit',
         prompt='### 영어: {text}</끝>\n### 한국어:', stop=['</끝>', '###']),
    dict(short='Rosetta-4B',           kind='llm', hf='yanolja/YanoljaNEXT-Rosetta-4B',       license='Gemma',        enabled=True, json_io=True,
         system_prompt="Translate the user's text to Korean.\nTone: keep the reviewer's sentiment and its intensity exactly.\n"
                       'Provide the final translation immediately without any other text.'),
    dict(short='EXAONE-3.5-7.8B',      kind='llm', hf='LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct', license='EXAONE NC',    enabled=True, quant='4bit',
         trust_remote_code=True),
    dict(short='Qwen2.5-7B',           kind='llm', hf='Qwen/Qwen2.5-7B-Instruct',             license='Apache-2.0',   enabled=True, quant='4bit'),
    dict(short='iris-7B',              kind='llm', hf='davidkim205/iris-7b',                  license='Apache-2.0',   enabled=True, quant='4bit',
         prompt='[INST] 다음 문장을 한글로 번역하세요.{text} [/INST]', stop=['</s>', '[INST]']),
    dict(short='llama3-instrucTrans-8B', kind='llm', hf='nayohan/llama3-instrucTrans-enko-8b', license='Llama 3',      enabled=True, quant='4bit',
         system_prompt='당신은 번역기 입니다. 영어를 한국어로 번역하세요.'),
    # 꺼 둔 후보: 크거나(시간), 게이트가 있거나, 같은 계열이 이미 켜져 있음
    dict(short='EXAONE-3.5-2.4B',      kind='llm', hf='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct', license='EXAONE NC',    enabled=False, trust_remote_code=True),
    dict(short='Qwen3-4B',             kind='llm', hf='Qwen/Qwen3-4B-Instruct-2507',          license='Apache-2.0',   enabled=False),
    dict(short='Qwen3-8B',             kind='llm', hf='Qwen/Qwen3-8B',                        license='Apache-2.0',   enabled=False, quant='4bit'),
    dict(short='Rosetta-12B',          kind='llm', hf='yanolja/YanoljaNEXT-Rosetta-12B',      license='Gemma',        enabled=False, quant='4bit', json_io=True,
         system_prompt="Translate the user's text to Korean.\nTone: keep the reviewer's sentiment and its intensity exactly.\n"
                       'Provide the final translation immediately without any other text.'),
    dict(short='TowerInstruct-7B',     kind='llm', hf='Unbabel/TowerInstruct-7B-v0.2',        license='CC-BY-NC-4.0', enabled=False, quant='4bit',
         system_prompt=None, user_template='Translate the following text from English into Korean.\nEnglish: {text}\nKorean:'),
    dict(short='Tower-Plus-9B',        kind='llm', hf='Unbabel/Tower-Plus-9B',                license='CC-BY-NC-SA-4.0', enabled=False, quant='4bit',
         system_prompt=None, user_template='Translate the following English source text to Korean:\nEnglish: {text}\nKorean: '),
]
MT_EVAL_N       = 300      # 번역 후보 비교에 쓸 테스트 리뷰 수 (별점별로 고르게)
MAIN_MT         = 'auto'   # 전체 번역에 쓸 후보 이름. 'auto'면 비교 점수 1위(아래 AUTO_ALLOW_API 참고)
AUTO_ALLOW_API  = False    # 'auto'에서 Gemini·Papago도 고를지. 2만 건 API 번역은 요청이 많아 기본은 오픈 모델 중에서만 고름
BACK_MT         = 'M2M100-1.2B'   # 역번역(한→영)에 쓸 모델 (위 목록의 이름, 오픈 모델만)
# 감성 유지 판정기: 영어와 한국어를 모두 읽는 다국어 감성 모델. 여러 개의 평균을 써서 한 모델의 버릇에 덜 휘둘리게 함
SENTI_JUDGES    = [
    dict(short='tabularisai', hf='tabularisai/multilingual-sentiment-analysis'),   # 5단계 (매우 부정 ~ 매우 긍정)
    dict(short='clapAI',      hf='clapAI/roberta-base-multilingual-sentiment'),    # 2단계 (부정/긍정)
]
USE_LLM_JUDGE   = True     # GEMINI_KEY가 있으면 Gemini가 원문·번역문의 별점을 따로 추정해 비교 (번역 후보 수 × 15번 정도 요청)
LLM_JUDGE_MODEL = 'gemini-3.5-flash-lite'
USE_COMETKIWI   = False    # 참조 없는 번역 품질 점수(CometKiwi). HF에서 약관 동의 + HF_TOKEN 필요

MT_BATCH        = 32       # 번역 전용 모델 배치 크기 (GPU 메모리가 부족하면 줄이세요)
LLM_BATCH       = 8        # 번역 LLM 배치 크기
LLM_MAX_NEW_TOKENS = 768   # 번역 LLM이 한 리뷰에 생성할 최대 토큰 수
MT_BEAMS        = 2        # 빔 서치 폭. 1이면 빠르고, 4면 느리지만 조금 더 자연스러움
MAX_SRC_CHARS   = 1200     # 원문을 이 길이에서 자름 (분류 모델은 앞부분 160토큰만 보므로 충분)
MAX_CHUNK_CHARS = 400      # 문장을 이 길이 이하 덩어리로 묶어 번역 (긴 입력에서 생기는 누락·반복 방지)
GEMINI_BATCH    = 20       # Gemini 요청 1번에 넣을 리뷰 수
GEMINI_SLEEP    = 4.0      # Gemini 요청 사이 대기(초). 무료 등급 요청 수 제한을 피하려는 값
SAVE_EVERY      = 1000     # 전체 번역 중 이 건수마다 드라이브에 중간 저장

# ===== 1-4. 한국어 감성 모델 =====
# zero_shot=False: 감성 모델이 아닌 사전학습 모델이라 학습 전 평가를 건너뜀
MODELS = [
    dict(short='KcELECTRA',              hf='beomi/KcELECTRA-base',                         base='ELECTRA(한국어)',  license='MIT',          zero_shot=False),
    dict(short='KLUE-RoBERTa',           hf='klue/roberta-base',                            base='RoBERTa(한국어)',  license='카드 미표기',  zero_shot=False),
    dict(short='nlptown-mBERT',          hf='nlptown/bert-base-multilingual-uncased-sentiment', base='mBERT',        license='MIT'),
    dict(short='tabularisai-DistilBERT', hf='tabularisai/multilingual-sentiment-analysis',  base='DistilBERT',       license='CC-BY-NC-4.0'),
    dict(short='clapAI-XLMR',            hf='clapAI/roberta-base-multilingual-sentiment',   base='XLM-R',            license='Apache-2.0'),
]

# ===== 1-5. 학습 하이퍼파라미터 (이전 노트북과 같게) =====
MAX_LEN, BATCH_SIZE, EVAL_BATCH_SIZE = 160, 32, 128
EPOCHS, LR, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 3, 2e-5, 0.01, 0.1, 1
SAVE_MODELS = False
DASHBOARD_MAX_ROWS = 3000

In [4]:
# ── 폴더 준비, 비밀값 읽기 ───────────────────────────────────
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PREV_DIR, OUT_DIR = Path(DRIVE_ROOT) / PREV_DIR_NAME, Path(DRIVE_ROOT) / OUT_DIR_NAME
else:
    PREV_DIR, OUT_DIR = Path(PREV_DIR_NAME), Path(OUT_DIR_NAME)

RESULTS_DIR = OUT_DIR / 'results'        # 한국어 감성 모델별 결과(JSON)
TRANS_DIR   = OUT_DIR / 'translations'   # 번역 후보별 번역문 캐시(parquet)
MODELS_DIR  = OUT_DIR / 'models'
for d in (OUT_DIR, RESULTS_DIR, TRANS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def read_secret(name):
    """환경변수 → Colab Secrets 순서로 비밀값을 찾는다. 없으면 None."""
    if os.environ.get(name):
        return os.environ[name]
    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            return None
    return None


GEMINI_KEY = read_secret('GEMINI_KEY')
if read_secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = read_secret('HF_TOKEN')

PAPAGO_ID, PAPAGO_SECRET = read_secret('PAPAGO_CLIENT_ID'), read_secret('PAPAGO_CLIENT_SECRET')

# 키가 없는 API 후보는 자동으로 뺌
for c in MT_CANDIDATES:
    if c['enabled'] and ((c['kind'] == 'gemini' and not GEMINI_KEY) or
                         (c['kind'] == 'papago' and not (PAPAGO_ID and PAPAGO_SECRET))):
        c['enabled'] = False
        print(f"API 키가 없어 {c['short']}는 비교에서 뺍니다.")

MT_BY_NAME = {c['short']: c for c in MT_CANDIDATES}
print('이전 결과 폴더:', PREV_DIR.resolve())
print('이번 결과 폴더:', OUT_DIR.resolve())
print('비교할 번역 후보:', [c['short'] for c in MT_CANDIDATES if c['enabled']])

Mounted at /content/drive
API 키가 없어 Papago는 비교에서 뺍니다.
이전 결과 폴더: /content/drive/MyDrive/voc_amazon_finetune
이번 결과 폴더: /content/drive/MyDrive/voc_amazon_ko
비교할 번역 후보: ['Gemini-3.5-FlashLite', 'NLLB-600M', 'NLLB-1.3B', 'NHNDQ-NLLB-en2ko', 'M2M100-418M', 'M2M100-1.2B', 'Gugugo-7B', 'Rosetta-4B', 'EXAONE-3.5-7.8B', 'Qwen2.5-7B', 'iris-7B', 'llama3-instrucTrans-8B']


## 2. 이전 실험의 샘플과 분할 재현
이전 노트북과 **똑같은 정제·분할 코드**를 같은 시드로 돌립니다. 그래야 영어 결과와 한국어 결과를 같은 테스트 리뷰로 비교할 수 있습니다.
마지막에 이전 결과 파일의 테스트 리뷰 ID와 같은지 확인합니다.

In [5]:
SAMPLE_PATH = PREV_DIR / 'sample_raw.parquet'
assert SAMPLE_PATH.exists(), f'{SAMPLE_PATH}가 없습니다. 이전 노트북을 먼저 실행하거나 PREV_DIR_NAME을 확인하세요.'
raw = pd.read_parquet(SAMPLE_PATH)

STAR2SENT  = {1: '부정', 2: '부정', 3: '중립', 4: '긍정', 5: '긍정'}
SENT_ORDER = ['부정', '중립', '긍정']
STARS      = [1, 2, 3, 4, 5]

# ↓ 이전 노트북 3장과 같은 코드 (바꾸면 테스트셋이 달라집니다)
df = raw.copy()
df['input_text'] = np.where(df['title'].str.len() > 0, df['title'] + '. ' + df['text'], df['text'])
df['input_text'] = df['input_text'].str.replace(r'\s+', ' ', regex=True).str.strip()
df = df.drop_duplicates('input_text').drop_duplicates('review_id')
df = (df.sample(frac=1, random_state=SEED)
        .groupby(['category', 'star']).head(N_PER_CLASS)
        .reset_index(drop=True))
df['sentiment'] = df['star'].map(STAR2SENT)
df['label']     = df['star'] - 1
df['date']      = pd.to_datetime(df['timestamp'], unit='ms').dt.strftime('%Y-%m-%d')
df['strata']    = df['category'] + '_' + df['star'].astype(str)

train_df, rest_df = train_test_split(df, test_size=1 - SPLIT[0], stratify=df['strata'], random_state=SEED)
val_df, test_df   = train_test_split(rest_df, test_size=SPLIT[2] / (SPLIT[1] + SPLIT[2]),
                                     stratify=rest_df['strata'], random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))
print(f'학습 {len(train_df):,} / 검증 {len(val_df):,} / 테스트 {len(test_df):,}건')

# 이전 영어 결과 불러오기 (테스트 리뷰 ID가 같을 때만 비교에 사용)
en_results = {}
for p in sorted((PREV_DIR / 'results').glob('*.json')):
    r = json.loads(p.read_text())
    if r.get('review_ids') == test_df['review_id'].tolist():
        en_results[r['short']] = r
    else:
        print(f'[제외] {p.name}: 테스트셋이 지금과 다름')
print('비교에 쓸 영어 결과:', list(en_results) or '없음')

학습 13,999 / 검증 3,000 / 테스트 3,001건
비교에 쓸 영어 결과: ['cardiff-XLMR', 'lxyuan-DistilBERT', 'nlptown-mBERT', 'tabularisai-DistilBERT']


## 3. 번역 전 원문 정리
- 아마존 본문에 섞인 `<br />` 같은 HTML 태그와 `&amp;` 같은 엔티티를 지웁니다.
- 너무 긴 리뷰는 `MAX_SRC_CHARS`에서 문장 끝 기준으로 자릅니다.
- 번역 전용 모델은 문장 단위로 학습돼서 긴 입력에서 뒷부분을 빼먹거나 같은 말을 반복하기 쉽습니다. 그래서 문장을 `MAX_CHUNK_CHARS` 이하 덩어리로 묶어 따로 번역한 뒤 다시 이어 붙입니다.

In [6]:
def clean_source(text):
    """번역 입력용으로 원문을 정리한다: HTML 제거, 공백 정리, 길이 제한."""
    t = html.unescape(str(text))
    t = re.sub(r'<br\s*/?>', ' ', t, flags=re.I)        # 줄바꿈 태그
    t = re.sub(r'<[^>]{1,40}>', ' ', t)                  # 그 밖의 짧은 HTML 태그
    t = re.sub(r'\s+', ' ', t).strip()
    if len(t) > MAX_SRC_CHARS:
        cut = max(t.rfind(p, 0, MAX_SRC_CHARS) for p in '.!?')
        t = t[:cut + 1] if cut > MAX_SRC_CHARS // 2 else t[:MAX_SRC_CHARS]
    return t


def split_chunks(text, max_chars=MAX_CHUNK_CHARS):
    """문장 단위로 나눈 뒤 max_chars 이하가 되게 이어 붙인 덩어리 목록을 돌려준다.
    마침표 없이 아주 긴 문장은 공백 위치에서 강제로 자른다."""
    sents = re.split(r'(?<=[.!?…])\s+', text.strip())
    chunks, cur = [], ''
    for s in sents:
        while len(s) > max_chars:                         # 혼자서도 너무 긴 문장
            cut = s.rfind(' ', 0, max_chars)
            cut = cut if cut > max_chars // 2 else max_chars
            if cur:
                chunks.append(cur); cur = ''
            chunks.append(s[:cut].strip()); s = s[cut:].strip()
        if not s:
            continue
        if cur and len(cur) + 1 + len(s) > max_chars:
            chunks.append(cur); cur = s
        else:
            cur = f'{cur} {s}'.strip()
    if cur:
        chunks.append(cur)
    return chunks or ['']


df['src_en'] = df['input_text'].map(clean_source)
for d in (train_df, val_df, test_df):
    d['src_en'] = d['input_text'].map(clean_source)

n_chunks = df['src_en'].map(lambda t: len(split_chunks(t)))
print('정리 후 원문 길이(자) 분포:'); display(df['src_en'].str.len().describe().round(0).to_frame('글자 수').T)
print(f'리뷰당 번역 덩어리 수: 평균 {n_chunks.mean():.2f}, 최대 {n_chunks.max()}')
print('예시:', split_chunks(df.loc[df['src_en'].str.len().idxmax(), 'src_en'])[:3])

정리 후 원문 길이(자) 분포:


,count,mean,std,min,25%,50%,75%,max
글자 수,20000.0,213.0,187.0,24.0,90.0,156.0,269.0,1200.0


리뷰당 번역 덩어리 수: 평균 1.15, 최대 4
예시: ["Great product but be very careul on size. I love this product: Vapor Apparel Men's UPF 50+ UV Sun Protection Performance Long Sleeve T-Shirt. But please be cautious about finding YOUR SIZE. I carefully measured my chest and waist circumferences when I inhaled deeply and when I exhaled completely. According to the size chart, I fit reasonably well within their Medium category.", "Normally, I wear a Small size, but I decided to trust the chart. Please don't. The Medium size was very baggy and with extra long sleeves. Then I exchanged this product for the Small size. Better, but not good enough: still not even close to T-shirts in my wardrobe. Then I exchanged the Small size for an extra small (XS). At last a perfect fit.", 'I do not understand why more reviewers did not report the sizing problem I have experienced. Kudos to Amazon and Vapor Apparel for allowing me to make these exchanges. All the rest of my comments are very positive. I now have both the

## 4. 공통 함수: 평가 지표와 모델 실행
이전 노트북과 같은 지표·학습 코드입니다. 입력 열만 `TEXT_COL`(기본: 한국어 번역문 `ko_text`)로 바꿨습니다.

In [7]:
def native_label_to_star(label):
    """모델의 원래 라벨 이름을 별점(1~5)으로 바꾼다. 별점으로 바꿀 수 없으면 None."""
    s = str(label).lower().strip()
    if 'star' in s:                       # nlptown: '1 star', '5 stars'
        digits = [c for c in s if c.isdigit()]
        return int(digits[0]) if digits else None
    five = {'very negative': 1, 'negative': 2, 'neutral': 3, 'positive': 4, 'very positive': 5}
    return five.get(s)                    # tabularisai: Very Negative ~ Very Positive


def native_label_to_sent(label):
    """모델의 원래 라벨 이름을 부정/중립/긍정으로 바꾼다 (제로샷 평가용)."""
    s = str(label).lower()
    if 'star' in s:
        star = native_label_to_star(s)
        return STAR2SENT.get(star)
    if 'neg' in s:  return '부정'         # 'negative', 'very negative'
    if 'neu' in s:  return '중립'
    if 'pos' in s:  return '긍정'         # 'positive', 'very positive'
    return None


def compute_metrics(y_true, y_pred, categories):
    """실제 별점·예측 별점·카테고리 배열을 받아 모든 지표를 dict로 돌려준다."""
    y_true, y_pred, categories = np.asarray(y_true), np.asarray(y_pred), np.asarray(categories)
    sent_true = np.array([STAR2SENT[s] for s in y_true])
    sent_pred = np.array([STAR2SENT[s] for s in y_pred])
    m = dict(
        accuracy=float(accuracy_score(y_true, y_pred)),
        macro_f1=float(f1_score(y_true, y_pred, labels=STARS, average='macro', zero_division=0)),
        within1=float(np.mean(np.abs(y_true - y_pred) <= 1)),
        mae=float(mean_absolute_error(y_true, y_pred)),
        sent_accuracy=float(accuracy_score(sent_true, sent_pred)),
        per_star={int(s): float(np.mean(y_pred[y_true == s] == s)) for s in STARS if (y_true == s).any()},
        per_category={str(c): float(np.mean(y_pred[categories == c] == y_true[categories == c]))
                      for c in pd.unique(categories)},
        per_sentiment={s: float(np.mean(sent_pred[sent_true == s] == s)) for s in SENT_ORDER if (sent_true == s).any()},
        confusion=confusion_matrix(y_true, y_pred, labels=STARS).tolist(),   # 행 = 실제, 열 = 예측
    )
    return m


def zero_shot_metrics(native_labels, y_true):
    """학습 전 모델의 원래 출력으로 감성(3분류)·별점 정확도를 잰다.
    clapAI처럼 긍정/부정만 있는 모델은 중립을 낼 수 없어서 감성 정확도가 구조적으로 낮게 나옵니다."""
    y_true = np.asarray(y_true)
    sent_true = np.array([STAR2SENT[s] for s in y_true])
    sent_pred = np.array([native_label_to_sent(l) for l in native_labels], dtype=object)
    stars = [native_label_to_star(l) for l in native_labels]
    out = dict(zs_sent_accuracy=float(np.mean(sent_pred == sent_true)))
    # 원래 출력이 5단계(별점으로 바꿀 수 있는 경우)인 모델만 별점 정확도도 계산
    if all(s is not None for s in stars):
        out['zs_star_accuracy'] = float(np.mean(np.array(stars) == y_true))
    return out

In [8]:
TEXT_COL = 'ko_text'   # 분류 모델이 읽을 열 (한국어 번역문)
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'          # GPU면 16비트 혼합정밀도(AMP)로 학습 속도를 높임
print('장치:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(GPU 없음: 매우 느립니다)')

ID2LABEL = {i: f'{i + 1} star' for i in range(5)}   # 0 → '1 star', ..., 4 → '5 star'
LABEL2ID = {v: k for k, v in ID2LABEL.items()}


def set_seed(seed):
    """재현성을 위해 모든 난수 시드를 고정한다."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loader(tok, texts, labels=None, batch_size=32, shuffle=False):
    """문장 목록을 배치로 묶어 토큰화하는 DataLoader를 만든다.
    배치마다 그 배치에서 가장 긴 문장 길이에 맞춰 패딩해서(동적 패딩) 계산을 줄인다."""
    items = list(zip(texts, labels)) if labels is not None else [(t, None) for t in texts]

    def collate(batch):
        enc = tok([b[0] for b in batch], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors='pt')
        if labels is not None:
            enc['labels'] = torch.tensor([b[1] for b in batch], dtype=torch.long)
        return enc

    g = torch.Generator().manual_seed(SEED)
    return DataLoader(items, batch_size=batch_size, shuffle=shuffle, collate_fn=collate, generator=g)


@torch.inference_mode()
def predict_probs(model, tok, texts):
    """문장 목록에 대한 클래스별 확률(softmax)을 numpy 배열로 돌려준다."""
    model.eval()
    out = []
    for batch in make_loader(tok, texts, batch_size=EVAL_BATCH_SIZE):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        out.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
    return np.concatenate(out)


def free_memory():
    """쓰지 않는 객체를 정리하고 GPU 메모리를 비운다 (모델 변수를 del 한 뒤 호출)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def zero_shot(cfg, texts):
    """학습 전 원래 모델로 예측해 원래 라벨 이름 목록을 돌려준다."""
    tok = AutoTokenizer.from_pretrained(cfg['hf'])
    model = AutoModelForSequenceClassification.from_pretrained(cfg['hf']).to(DEVICE)
    probs = predict_probs(model, tok, texts)
    id2label = model.config.id2label
    override = cfg.get('label_override', {})
    labels = [override.get(id2label[i], id2label[i]) for i in probs.argmax(1)]
    del model
    free_memory()
    return labels


def fine_tune(cfg):
    """모델 하나를 파인튜닝하고, 테스트 예측과 학습 기록을 돌려준다."""
    set_seed(SEED)
    tok = AutoTokenizer.from_pretrained(cfg['hf'])
    # num_labels=5로 분류 헤드를 별점 5개로 맞춤.
    # 원래 헤드 크기가 다르면(3개, 2개) ignore_mismatched_sizes=True로 헤드만 새로 만들고 나머지 가중치는 그대로 씀.
    # nlptown처럼 원래도 5개면 기존 헤드를 그대로 이어서 학습함.
    # 가중치는 32비트로 불러옴: 16비트로 저장된 모델(clapAI)은 그대로 두면 'unscale FP16 gradients' 오류가 남.
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg['hf'], num_labels=5, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True, **{DTYPE_KW: torch.float32}).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())

    train_loader = make_loader(tok, train_df[TEXT_COL].tolist(), train_df['label'].tolist(),
                               batch_size=BATCH_SIZE, shuffle=True)
    total_steps  = EPOCHS * len(train_loader)
    warmup_steps = int(WARMUP_RATIO * total_steps)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    # 학습률 스케줄: 워밍업 동안 0→LR로 올리고, 이후 끝까지 직선으로 0까지 내림
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)   # 16비트 학습에서 기울기 값이 사라지지 않게 보정

    history, best_f1, best_state, best_epoch, bad_epochs = [], -1.0, None, 0, 0
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for batch in tqdm(train_loader, desc=f"{cfg['short']} epoch {epoch}", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                loss = model(**batch).loss                  # labels를 넣으면 교차 엔트로피 손실을 계산해 줌
            scaler.scale(loss).backward()                   # 역전파
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 기울기 폭주 방지
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            running += loss.item()

        # 에폭이 끝날 때마다 검증셋으로 성능 확인
        val_pred = predict_probs(model, tok, val_df[TEXT_COL].tolist()).argmax(1) + 1
        val_acc = float(accuracy_score(val_df['star'], val_pred))
        val_f1  = float(f1_score(val_df['star'], val_pred, labels=STARS, average='macro', zero_division=0))
        history.append(dict(epoch=epoch, train_loss=running / len(train_loader), val_accuracy=val_acc, val_macro_f1=val_f1))
        print(f"  epoch {epoch}: train_loss={history[-1]['train_loss']:.4f}  val_acc={val_acc:.4f}  val_macroF1={val_f1:.4f}")

        if val_f1 > best_f1:                                # 가장 좋은 시점의 가중치를 CPU 메모리에 복사해 둠
            best_f1, best_epoch, bad_epochs = val_f1, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:                      # 조기 종료
                print('  검증 성능이 나아지지 않아 조기 종료합니다.')
                break
    train_minutes = (time.time() - t0) / 60

    model.load_state_dict(best_state)                       # 가장 좋았던 가중치로 되돌림
    t1 = time.time()
    probs = predict_probs(model, tok, test_df[TEXT_COL].tolist())
    infer_sec = time.time() - t1

    if SAVE_MODELS:
        save_dir = MODELS_DIR / cfg['short']
        model.save_pretrained(save_dir); tok.save_pretrained(save_dir)

    del model, best_state, optimizer, scaler
    free_memory()
    return dict(n_params=int(n_params), history=history, best_epoch=best_epoch,
                train_minutes=train_minutes, infer_per_sec=len(test_df) / infer_sec,
                test_pred=(probs.argmax(1) + 1).tolist(),          # 예측 별점 (1~5)
                test_conf=probs.max(1).round(4).tolist(),          # 확신도 = 가장 높은 확률
                test_probs=probs.round(4).tolist())

장치: cuda Tesla T4


## 5. 번역기
후보 종류마다 언어 지정 방식이 달라서 하나의 함수 `translate()`로 감쌌습니다.

| 종류 | 한국어로 번역시키는 방법 |
|---|---|
| NLLB | 토크나이저에 `src_lang='eng_Latn'`, 생성 시 첫 토큰을 `kor_Hang`으로 고정 |
| M2M100 | `tokenizer.src_lang='en'`, 생성 시 첫 토큰을 `ko` 언어 토큰으로 고정 |
| OPUS-MT | 입력 앞에 `>>kor<<` |
| MADLAD | 입력 앞에 `<2ko>` |
| 번역 LLM | 채팅 템플릿에 감성 유지 지시문 + 리뷰, 또는 모델 카드의 고정 형식(예: Gugugo `### 영어: …</끝>\n### 한국어:`) |
| Gemini | 20건씩 JSON으로 보내고 JSON으로 받음 |
| Papago | 리뷰 한 건씩 API 호출 |

번역 전용 모델은 문장 덩어리로 나눠 번역하고, LLM과 API는 비꼼이나 앞뒤 대비 같은 문맥을 살리려고 리뷰를 통째로 번역합니다.

번역문은 후보마다 `translations/후보이름.parquet`에 리뷰 ID와 함께 저장해서, 같은 리뷰를 두 번 번역하지 않습니다.

In [9]:
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoModelForCausalLM

# transformers 4.56부터 torch_dtype 대신 dtype을 씁니다. 두 버전 모두에서 돌도록 맞춤.
_tv = tuple(int(x) for x in re.findall(r'\d+', transformers.__version__)[:2])
DTYPE_KW = 'dtype' if _tv >= (4, 56) else 'torch_dtype'
print('transformers', transformers.__version__)

LANG_CODES = {   # 종류별 (영어 코드, 한국어 코드)
    'nllb': ('eng_Latn', 'kor_Hang'), 'm2m': ('en', 'ko'), 'madlad': ('<2en>', '<2ko>'), 'marian': (None, '>>kor<<'),
}


def m2m_lang_id(tok, lang):
    """M2M100 언어 토큰 ID (버전에 따라 get_lang_id가 없을 수 있어 대비)."""
    if hasattr(tok, 'get_lang_id'):
        return tok.get_lang_id(lang)
    return tok.convert_tokens_to_ids(f'__{lang}__')


class Seq2SeqTranslator:
    """번역 전용 모델(NLLB, M2M100, OPUS-MT, MADLAD). direction은 'en2ko' 또는 'ko2en'."""

    def __init__(self, cfg, direction='en2ko'):
        self.cfg, self.kind = cfg, cfg['kind']
        src, tgt = LANG_CODES[self.kind] if direction == 'en2ko' else LANG_CODES[self.kind][::-1]
        if self.kind == 'marian' and direction != 'en2ko':
            raise ValueError('OPUS-MT en-ko 모델은 영→한 전용입니다.')
        self.src, self.tgt = src, tgt
        tok_kwargs = {'src_lang': src} if self.kind == 'nllb' else {}
        self.tok = AutoTokenizer.from_pretrained(cfg['hf'], **tok_kwargs)
        dtype = torch.float16 if (DEVICE == 'cuda' and not cfg.get('fp32')) else torch.float32
        self.model = AutoModelForSeq2SeqLM.from_pretrained(cfg['hf'], **{DTYPE_KW: dtype}).to(DEVICE).eval()
        self.gen_kwargs = {}
        if self.kind == 'nllb':
            self.gen_kwargs['forced_bos_token_id'] = self.tok.convert_tokens_to_ids(tgt)
        elif self.kind == 'm2m':
            self.tok.src_lang = src
            self.gen_kwargs['forced_bos_token_id'] = m2m_lang_id(self.tok, tgt)

    def _prep(self, text):
        if self.kind == 'marian':
            return f'{self.tgt} {text}'
        if self.kind == 'madlad':
            return f'{self.tgt} {text}'
        return text

    @torch.inference_mode()
    def translate_texts(self, texts, desc=''):
        """짧은 문장 목록을 번역한다. 길이순으로 정렬해 배치마다 패딩을 줄이고, 원래 순서로 되돌린다."""
        order = np.argsort([len(t) for t in texts])[::-1]
        out = [''] * len(texts)
        for i in tqdm(range(0, len(texts), MT_BATCH), desc=desc or self.cfg['short'], leave=False):
            idx = order[i:i + MT_BATCH]
            batch = [self._prep(texts[j]) for j in idx]
            enc = self.tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=512).to(DEVICE)
            max_new = int(min(512, enc['input_ids'].shape[1] * 2 + 16))   # 원문 길이에 맞춘 출력 상한
            gen = self.model.generate(**enc, num_beams=MT_BEAMS, max_new_tokens=max_new,
                                      no_repeat_ngram_size=0, **self.gen_kwargs)
            for j, t in zip(idx, self.tok.batch_decode(gen, skip_special_tokens=True)):
                out[j] = t.strip()
        return out

    def close(self):
        del self.model
        free_memory()


LLM_INSTRUCTION = (
    'Translate the following English product review into natural Korean, the way a Korean online shopper '
    'would write a review. Preserve the sentiment exactly: keep the polarity and intensity of praise and '
    'complaints, sarcasm, hedging, and emphasis (ALL CAPS, "!!!"). Do not add, omit, soften, or explain '
    'anything. Keep brand names and sizes as they are. Output only the Korean translation.')
LABEL_PREFIX = re.compile(r'^\s*(번역|한국어|Korean|Translation)\s*[:：]\s*', re.I)


class LLMTranslator:
    """번역 LLM (Gugugo, EXAONE, Qwen, Rosetta 등). 리뷰를 통째로 넣고 생성 결과에서 번역문만 잘라 낸다.
    cfg['prompt']가 'chat'이면 모델의 채팅 템플릿에 LLM_INSTRUCTION과 리뷰를 넣고,
    그 밖의 문자열이면 '{text}' 자리에 리뷰를 넣은 고정 형식(예: Gugugo)으로 쓴다."""
    chunked = False

    def __init__(self, cfg, direction='en2ko'):
        if direction != 'en2ko':
            raise ValueError('번역 LLM은 영→한 비교에만 씁니다.')
        self.cfg = cfg
        trc = cfg.get('trust_remote_code', False)
        self.tok = AutoTokenizer.from_pretrained(cfg['hf'], trust_remote_code=trc)
        self.tok.padding_side = 'left'                       # 생성 모델은 왼쪽 패딩
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = dict(trust_remote_code=trc)
        if cfg.get('quant') == '4bit' and DEVICE == 'cuda':
            # 7B 이상은 T4(16GB)에 16비트로 안 들어가서 4비트로 줄여 올림 (품질은 조금 떨어질 수 있음)
            from transformers import BitsAndBytesConfig
            kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                           bnb_4bit_compute_dtype=torch.float16)
            kw['device_map'] = 'auto'
            self.model = AutoModelForCausalLM.from_pretrained(cfg['hf'], **kw).eval()
        else:
            kw[DTYPE_KW] = torch.float16 if DEVICE == 'cuda' else torch.float32
            self.model = AutoModelForCausalLM.from_pretrained(cfg['hf'], **kw).to(DEVICE).eval()

    def _prompt(self, text):
        if self.cfg.get('json_io'):                          # Rosetta: 원문을 JSON 문자열로 보내라는 카드 예시를 따름
            text = json.dumps(text, ensure_ascii=False)
        if self.cfg.get('prompt', 'chat') == 'chat':
            system = self.cfg.get('system_prompt', LLM_INSTRUCTION)
            user = self.cfg.get('user_template', '{text}').replace('{text}', text)
            msgs = ([{'role': 'system', 'content': system}] if system else []) + [{'role': 'user', 'content': user}]
            try:                                             # Qwen3 같은 모델의 '생각' 출력 끄기
                return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                                    enable_thinking=False)
            except TypeError:
                return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return (self.tok.bos_token or '') + self.cfg['prompt'].replace('{text}', text)   # 고정 형식은 BOS를 직접 붙임

    def _clean(self, t):
        for s in self.cfg.get('stop', []):                   # 멈춤 문자열 뒤는 버림
            t = t.split(s)[0]
        t = re.sub(r'<think>.*?</think>', '', t, flags=re.S)
        t = LABEL_PREFIX.sub('', t.strip())
        if self.cfg.get('json_io'):
            try:
                v = json.loads(t)
                if isinstance(v, str):
                    return v.strip()                         # JSON 문자열이면 따옴표 처리가 끝난 값 그대로
                if isinstance(v, dict):
                    t = ' '.join(map(str, v.values()))
            except Exception:
                pass
        return t.strip().strip('"“”').strip()

    @torch.inference_mode()
    def translate_texts(self, texts, desc=''):
        out = [''] * len(texts)
        order = np.argsort([len(t) for t in texts])[::-1]
        dev = self.model.device
        for i in tqdm(range(0, len(texts), LLM_BATCH), desc=desc or self.cfg['short'], leave=False):
            idx = order[i:i + LLM_BATCH]
            prompts = [self._prompt(texts[j]) for j in idx]
            enc = self.tok(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(dev)
            src_len = max(len(self.tok(texts[j], add_special_tokens=False)['input_ids']) for j in idx)
            gen = self.model.generate(**enc, max_new_tokens=int(min(LLM_MAX_NEW_TOKENS, src_len * 3 + 32)),
                                      do_sample=False, repetition_penalty=1.05,
                                      pad_token_id=self.tok.pad_token_id)
            new_tokens = gen[:, enc['input_ids'].shape[1]:]      # 프롬프트 뒤에 새로 만든 부분만
            for j, t in zip(idx, self.tok.batch_decode(new_tokens, skip_special_tokens=True)):
                out[j] = self._clean(t)
        return out

    def close(self):
        del self.model
        free_memory()


class GeminiClient:
    """Gemini API를 JSON 입출력으로 부르는 공통 도우미 (번역과 별점 판정에 같이 씀)."""

    def __init__(self, model):
        from google import genai
        from google.genai import types
        self.model, self.types = model, types
        self.client = genai.Client(api_key=GEMINI_KEY)

    def call(self, prompt, items, key, tries=5):
        """items([{'id':..., ...}])를 보내고 {id: 응답[key]}를 돌려준다. 실패하면 빈 dict."""
        for k in range(tries):
            try:
                resp = self.client.models.generate_content(
                    model=self.model, contents=prompt + json.dumps(items, ensure_ascii=False),
                    config=self.types.GenerateContentConfig(response_mime_type='application/json', temperature=0))
                data = json.loads(resp.text)
                return {int(d['id']): d.get(key) for d in data if 'id' in d}
            except Exception as e:                     # 요청 수 제한(429), 일시 오류, JSON 깨짐
                wait = GEMINI_SLEEP * (2 ** k)
                print(f'  Gemini 재시도 {k + 1}/{tries} ({type(e).__name__}): {wait:.0f}초 대기')
                time.sleep(wait)
        return {}

    def batched(self, prompt, texts, field, key, desc=''):
        """texts를 GEMINI_BATCH건씩 보내고 빠진 건은 한 건씩 다시 보낸다."""
        out = [None] * len(texts)
        for i in tqdm(range(0, len(texts), GEMINI_BATCH), desc=desc, leave=False):
            items = [{'id': j, field: texts[j]} for j in range(i, min(i + GEMINI_BATCH, len(texts)))]
            got = self.call(prompt, items, key)
            for it in items:
                out[it['id']] = got.get(it['id'])
            time.sleep(GEMINI_SLEEP)
        for j in [j for j, v in enumerate(out) if v in (None, '')]:
            out[j] = self.call(prompt, [{'id': j, field: texts[j]}], key).get(j)
            time.sleep(GEMINI_SLEEP)
        return out


class GeminiTranslator:
    """Gemini API 번역기. 리뷰를 통째로 GEMINI_BATCH건씩 JSON으로 묶어 보낸다."""
    chunked = False
    PROMPT = (
        '다음은 아마존 패션·의류 상품 리뷰 목록입니다. 각 리뷰를 한국 온라인 쇼핑몰 리뷰처럼 자연스러운 한국어로 번역하세요.\n'
        '규칙: 의미와 감정의 세기(불만, 칭찬, 비꼼)를 그대로 유지하고, 내용을 더하거나 빼지 마세요. '
        '브랜드·모델명은 그대로 두고, 사이즈·단위는 원문 표기를 유지하세요.\n'
        '출력: [{"id": 번호, "ko": "번역문"}] 형식의 JSON 배열만 출력하세요.\n\n입력:\n')

    def __init__(self, cfg, direction='en2ko'):
        self.cfg, self.client = cfg, GeminiClient(cfg['model'])

    def translate_texts(self, texts, desc=''):
        got = self.client.batched(self.PROMPT, texts, 'en', 'ko', desc or self.cfg['short'])
        return [str(t or '').strip() for t in got]

    def close(self):
        pass


class PapagoTranslator:
    """네이버 클라우드 Papago 번역 API. Colab Secrets에 PAPAGO_CLIENT_ID, PAPAGO_CLIENT_SECRET이 있어야 한다.
    리뷰 한 건씩 보낸다(요청당 최대 5,000자). 입력 글자 수만큼 요금이 나오니 주의."""
    chunked = False
    URL = 'https://papago.apigw.ntruss.com/nmt/v1/translation'

    def __init__(self, cfg, direction='en2ko'):
        self.cfg = cfg
        self.src, self.tgt = ('en', 'ko') if direction == 'en2ko' else ('ko', 'en')
        self.headers = {'X-NCP-APIGW-API-KEY-ID': PAPAGO_ID, 'X-NCP-APIGW-API-KEY': PAPAGO_SECRET,
                        'Content-Type': 'application/json'}

    def _one(self, text, tries=5):
        for k in range(tries):
            try:
                r = requests.post(self.URL, headers=self.headers, timeout=30,
                                  json={'source': self.src, 'target': self.tgt, 'text': text[:5000]})
                r.raise_for_status()
                j = r.json()
                res = j.get('message', {}).get('result', j)     # 응답이 message.result 아래에 오는 형식도 처리
                return str(res.get('translatedText', '')).strip()
            except Exception as e:
                wait = 2 ** k
                print(f'  Papago 재시도 {k + 1}/{tries} ({type(e).__name__}): {wait}초 대기')
                time.sleep(wait)
        return ''

    def translate_texts(self, texts, desc=''):
        return [self._one(t) for t in tqdm(texts, desc=desc or self.cfg['short'], leave=False)]

    def close(self):
        pass


def make_translator(cfg, direction='en2ko'):
    if cfg['kind'] == 'gemini':
        return GeminiTranslator(cfg, direction)
    if cfg['kind'] == 'papago':
        return PapagoTranslator(cfg, direction)
    if cfg['kind'] == 'llm':
        return LLMTranslator(cfg, direction)
    return Seq2SeqTranslator(cfg, direction)


def translate_reviews(translator, texts):
    """리뷰 목록을 번역한다. 번역 전용 모델은 덩어리로 나눠 번역한 뒤 이어 붙이고,
    LLM·API는 문맥(비꼼, 앞뒤 대비)을 살리기 위해 리뷰를 통째로 번역한다."""
    if not getattr(translator, 'chunked', True):
        return translator.translate_texts(texts)
    pieces, owner = [], []
    for i, t in enumerate(texts):
        for c in split_chunks(t):
            pieces.append(c); owner.append(i)
    done = translator.translate_texts(pieces)
    out = [[] for _ in texts]
    for i, t in zip(owner, done):
        out[i].append(t)
    return [' '.join(x).strip() for x in out]


def get_translations(cfg, frame, desc=''):
    """frame(review_id, src_en)의 번역문을 캐시에서 꺼내고, 없는 것만 번역해 캐시에 더한다.
    반환: review_id 순서에 맞춘 번역문 Series, 새로 번역한 건수, 걸린 시간(초)"""
    path = TRANS_DIR / f"{cfg['short']}.parquet"
    cache = pd.read_parquet(path) if path.exists() else pd.DataFrame(columns=['review_id', 'ko'])
    have = set(cache['review_id'])
    todo = frame[~frame['review_id'].isin(have)]
    secs = 0.0
    if len(todo):
        print(f"[{cfg['short']}] 번역 {len(todo):,}건 {desc}")
        tr = make_translator(cfg)
        try:
            for s in range(0, len(todo), SAVE_EVERY):        # SAVE_EVERY건마다 드라이브에 저장
                part = todo.iloc[s:s + SAVE_EVERY]
                t0 = time.time()
                ko = translate_reviews(tr, part['src_en'].tolist())
                secs += time.time() - t0
                cache = pd.concat([cache, pd.DataFrame({'review_id': part['review_id'].values, 'ko': ko})],
                                  ignore_index=True)
                cache.to_parquet(path, index=False)
                print(f'  저장 {min(s + SAVE_EVERY, len(todo)):,}/{len(todo):,}')
        finally:
            tr.close()
    ko = frame['review_id'].map(cache.drop_duplicates('review_id', keep='last').set_index('review_id')['ko'])
    return ko.fillna(''), len(todo), secs

transformers 5.17.0


## 6. 번역 후보 비교: 감성 유지 중심 (테스트 리뷰 `MT_EVAL_N`건)
이 프로젝트에서 번역의 목적은 **감성을 그대로 옮기는 것**입니다. 그래서 원문과 번역문을 같은 감성 판정기에 넣고, 감성이 얼마나 그대로 남는지를 여러 각도로 잽니다.
판정기는 영어와 한국어를 모두 읽는 다국어 감성 모델(`SENTI_JUDGES`)이고, `GEMINI_KEY`가 있으면 Gemini가 원문과 번역문의 별점을 각각 추정하는 판정도 더합니다.

| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| 감성 일치율 | 원문과 번역문의 판정(부정/중립/긍정)이 같은 비율. 판정기들의 평균 | 높을수록 |
| 극성 반전율 | 원문은 긍정인데 번역문은 부정(또는 반대)으로 뒤집힌 비율. 가장 치명적인 오류 | 낮을수록 |
| 강도 유지율 | 판정 점수(-1 매우 부정 ~ +1 매우 긍정)의 차이를 0~100으로 환산. "괜찮다"가 "최고다"로 바뀌는 것을 잡음 | 높을수록 |
| 분포 유사도 | 원문과 번역문의 감성 확률 분포가 얼마나 비슷한지(1 − JS 발산) | 높을수록 |
| LLM 별점 일치율 / 차이 | (선택) Gemini가 원문과 번역문만 보고 추정한 별점이 같은 비율과 평균 차이 | 일치율 높을수록, 차이 낮을수록 |
| 번역문 감성 정확도 | 판정기가 번역문만 보고 실제 별점의 감성을 맞힌 비율 (원문 기준값과 비교) | 높을수록 |
| 역번역 chrF | 번역문을 `BACK_MT`로 다시 영어로 옮겨 원문과 비교한 점수. 직역일수록 높게 나와서 비중을 낮춤 | 높을수록 |
| 실패율 | 빈 번역, 한글 비율 50% 미만, 반복, 길이 비정상 | 낮을수록 |

**감성 유지 점수** = (감성 일치율 + 강도 유지율 + 분포 유사도 + LLM 별점 일치율[있을 때]) 평균 − 2 × 극성 반전율
**종합 점수** = 0.85 × 감성 유지 점수 + 0.15 × 역번역 chrF. 실패율이 5%를 넘는 후보는 자동 선택에서 뺍니다.

In [10]:
# 별점별로 고르게 MT_EVAL_N건 고르기 (이전 실행과 같은 리뷰가 뽑혀 번역 캐시를 다시 씀)
per = max(1, MT_EVAL_N // 5)
mt_eval = (test_df.sample(frac=1, random_state=SEED)      # 섞은 뒤 별점마다 앞에서 per건
                  .groupby('star').head(per)
                  .reset_index(drop=True))
print(f'비교용 리뷰 {len(mt_eval)}건')

mt_out, mt_speed = {}, {}
SPEED_PATH = TRANS_DIR / '_speed.json'
speed_log = json.loads(SPEED_PATH.read_text()) if SPEED_PATH.exists() else {}
for cfg in [c for c in MT_CANDIDATES if c['enabled']]:
    try:
        ko, n_new, secs = get_translations(cfg, mt_eval, '(비교용)')
    except Exception as e:                       # 다운로드 실패, 메모리 부족, 권한(게이트) 등
        print(f"[실패] {cfg['short']}: {repr(e)[:300]}")
        free_memory()
        continue
    if n_new:
        speed_log[cfg['short']] = n_new / max(secs, 1e-6)
        SPEED_PATH.write_text(json.dumps(speed_log))
    mt_out[cfg['short']] = ko.tolist()
    mt_speed[cfg['short']] = speed_log.get(cfg['short'])
print('번역 완료:', list(mt_out))

비교용 리뷰 300건
[Gugugo-7B] 번역 300건 (비교용)


config.json:   0%|          | 0.00/699 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[실패] llama3-instrucTrans-8B: RuntimeError('Task error: File reconstruction error: IO Error: No space left on device (os error 28)')
번역 완료: ['Gemini-3.5-FlashLite', 'NLLB-600M', 'NLLB-1.3B', 'NHNDQ-NLLB-en2ko', 'M2M100-418M', 'M2M100-1.2B', 'Gugugo-7B', 'Rosetta-4B', 'Qwen2.5-7B']


In [11]:
HANGUL = re.compile(r'[가-힣]')
LATIN  = re.compile(r'[A-Za-z]')
REPEAT = re.compile(r'(\S{2,}(?:\s+\S+){0,2}?)(?:\s+\1){3,}')   # 같은 1~3단어가 4번 넘게 연속


def failure_reason(src, ko):
    """번역 실패로 볼 만한 이유를 돌려준다. 문제 없으면 ''."""
    if not ko.strip():
        return '빈 번역'
    h, l = len(HANGUL.findall(ko)), len(LATIN.findall(ko))
    if h / max(1, h + l) < 0.5:
        return '한글 비율 낮음'
    if REPEAT.search(ko):
        return '반복'
    ratio = len(ko) / max(1, len(src))
    if ratio < 0.15 or ratio > 1.5:      # 한국어는 보통 영어 글자 수의 절반 안팎
        return '길이 비정상'
    return ''


def label_values(labels):
    """판정기의 라벨마다 감성 점수(-1 ~ +1)를 정한다.
    5단계(별점, very negative~very positive)는 -1, -0.5, 0, 0.5, 1 / 3단계는 -1, 0, 1 / 2단계는 -1, 1."""
    if len(labels) == 5 and all(native_label_to_star(l) is not None for l in labels):
        return np.array([(native_label_to_star(l) - 3) / 2 for l in labels])
    return np.array([{'부정': -1.0, '중립': 0.0, '긍정': 1.0}.get(native_label_to_sent(l), 0.0) for l in labels])


def js_divergence(p, q):
    """두 확률 분포의 젠슨-섀넌 발산 (밑 2, 0~1). 행마다 계산."""
    p, q = np.clip(p, 1e-9, 1), np.clip(q, 1e-9, 1)
    m = (p + q) / 2
    kl = lambda a, b: (a * np.log2(a / b)).sum(1)
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


# ── (1) 다국어 감성 판정기로 원문·번역문의 감성 확률 구하기 ──
src = mt_eval['src_en'].tolist()
true_sent = np.array(mt_eval['sentiment'])
judged = {}                                   # 판정기 → {'labels': [...], 'EN': 확률, 번역모델: 확률}
for j in SENTI_JUDGES:
    try:
        jtok = AutoTokenizer.from_pretrained(j['hf'])
        jm = AutoModelForSequenceClassification.from_pretrained(j['hf']).to(DEVICE)
    except Exception as e:
        print(f"[판정기 실패] {j['short']}: {repr(e)[:200]}")
        continue
    labels = [jm.config.id2label[i] for i in range(len(jm.config.id2label))]
    res = {'labels': labels, 'EN': predict_probs(jm, jtok, src)}
    for name, ko in mt_out.items():
        res[name] = predict_probs(jm, jtok, ko)
    judged[j['short']] = res
    del jm
    free_memory()
print('판정기:', list(judged))

# ── (2) (선택) Gemini 별점 판정: 원문과 번역문을 따로 보고 별점 1~5를 추정 ──
STAR_PROMPT = ('다음은 상품 리뷰 목록입니다(영어 또는 한국어). 각 리뷰 작성자가 줬을 별점(1~5 정수)을 리뷰 내용만 보고 추정하세요.\n'
               '출력: [{"id": 번호, "star": 별점}] 형식의 JSON 배열만 출력하세요.\n\n입력:\n')
llm_star = {}
if USE_LLM_JUDGE and GEMINI_KEY:
    STAR_PATH = OUT_DIR / 'mt_llm_star.json'               # 다시 실행할 때 같은 판정을 반복하지 않게 저장
    llm_star = json.loads(STAR_PATH.read_text()) if STAR_PATH.exists() else {}
    gc_ = GeminiClient(LLM_JUDGE_MODEL)
    for name, texts in [('EN', src)] + list(mt_out.items()):
        if name in llm_star and len(llm_star[name]) == len(texts):
            continue
        got = gc_.batched(STAR_PROMPT, texts, 'review', 'star', desc=f'별점 판정 {name}')
        llm_star[name] = [int(v) if str(v).isdigit() and 1 <= int(v) <= 5 else None for v in got]
        STAR_PATH.write_text(json.dumps(llm_star))

# ── (3) 역번역 chrF ──
import sacrebleu
BACK_PATH = OUT_DIR / 'mt_back.json'
back = json.loads(BACK_PATH.read_text()) if BACK_PATH.exists() else {}
todo = [n for n in mt_out if n not in back or len(back[n]) != len(mt_out[n])]
if todo:
    try:
        btr = make_translator(MT_BY_NAME[BACK_MT], direction='ko2en')
        for name in todo:
            back[name] = translate_reviews(btr, mt_out[name])
        btr.close()
        BACK_PATH.write_text(json.dumps(back, ensure_ascii=False))
    except Exception as e:
        print('[역번역 실패]', repr(e)[:300])

# ── (4) (선택) CometKiwi ──
kiwi = {}
if USE_COMETKIWI:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unbabel-comet>=2.0.0'], check=True)
        from comet import download_model, load_from_checkpoint
        km = load_from_checkpoint(download_model('Unbabel/wmt22-cometkiwi-da'))
        for name, ko in mt_out.items():
            data = [{'src': s, 'mt': m} for s, m in zip(src, ko)]
            kiwi[name] = float(km.predict(data, batch_size=32, gpus=1 if DEVICE == 'cuda' else 0).system_score)
        del km
        free_memory()
    except Exception as e:
        print('[CometKiwi 실패]', repr(e)[:300])

# ── (5) 후보별 지표 계산 ──
def sentiment_metrics(name):
    agree, flip, inten, simil, acc, per_review = [], [], [], [], [], []
    for jn, res in judged.items():
        vals = label_values(res['labels'])
        p_en, p_ko = res['EN'], res[name]
        s_en, s_ko = p_en @ vals, p_ko @ vals                     # 감성 점수 (-1 ~ +1)
        lab_en = np.array([native_label_to_sent(res['labels'][i]) for i in p_en.argmax(1)])
        lab_ko = np.array([native_label_to_sent(res['labels'][i]) for i in p_ko.argmax(1)])
        agree.append(np.mean(lab_en == lab_ko))
        flip.append(np.mean(((s_en > 0.33) & (s_ko < -0.33)) | ((s_en < -0.33) & (s_ko > 0.33))))
        inten.append(1 - np.mean(np.abs(s_en - s_ko)) / 2)
        simil.append(1 - np.mean(js_divergence(p_en, p_ko)))
        acc.append(np.mean(lab_ko == true_sent))
        per_review.append(s_ko - s_en)
    m = dict(agree=np.mean(agree), flip=np.mean(flip), inten=np.mean(inten), simil=np.mean(simil),
             acc=np.mean(acc), shift=np.mean(per_review, axis=0) if per_review else np.zeros(len(src)))
    if 'EN' in llm_star and name in llm_star:
        pairs = [(a, b) for a, b in zip(llm_star['EN'], llm_star[name]) if a and b]
        m['llm_agree'] = np.mean([a == b for a, b in pairs]) if pairs else np.nan
        m['llm_diff'] = np.mean([abs(a - b) for a, b in pairs]) if pairs else np.nan
    return m


en_acc = np.mean([np.mean(np.array([native_label_to_sent(r['labels'][i]) for i in r['EN'].argmax(1)]) == true_sent)
                  for r in judged.values()]) if judged else np.nan
rows, shifts = [], {}
for name, ko in mt_out.items():
    fails = [failure_reason(s, k) for s, k in zip(src, ko)]
    m = sentiment_metrics(name)
    shifts[name] = m['shift']
    parts = [m['agree'], m['inten'], m['simil']] + ([m['llm_agree']] if not np.isnan(m.get('llm_agree', np.nan)) else [])
    keep = 100 * np.mean(parts) - 2 * 100 * m['flip']
    rows.append({
        '번역 모델': name, '종류': MT_BY_NAME[name]['kind'], '라이선스': MT_BY_NAME[name]['license'],
        '감성 유지 점수': round(keep, 1),
        '감성 일치율(%)': round(100 * m['agree'], 1), '극성 반전율(%)': round(100 * m['flip'], 1),
        '강도 유지율(%)': round(100 * m['inten'], 1), '분포 유사도(%)': round(100 * m['simil'], 1),
        'LLM 별점 일치율(%)': round(100 * m['llm_agree'], 1) if 'llm_agree' in m else None,
        'LLM 별점 차이': round(m['llm_diff'], 2) if 'llm_diff' in m else None,
        '번역문 감성 정확도(%)': round(100 * m['acc'], 1),
        '역번역 chrF': round(sacrebleu.corpus_chrf(back[name], [src]).score, 1) if name in back else None,
        'CometKiwi': round(kiwi[name], 3) if name in kiwi else None,
        '실패율(%)': round(100 * np.mean([f != '' for f in fails]), 1),
        '실패 유형': ', '.join(f'{k} {v}' for k, v in pd.Series([f for f in fails if f]).value_counts().items()) or '-',
        '속도(건/초)': round(mt_speed[name], 1) if mt_speed.get(name) else None,
    })
mt_compare = pd.DataFrame(rows)
mt_compare['종합 점수'] = (0.85 * mt_compare['감성 유지 점수'] + 0.15 * mt_compare['역번역 chrF'].fillna(0)).round(1)
mt_compare = mt_compare.sort_values('종합 점수', ascending=False).reset_index(drop=True)
mt_compare = mt_compare.dropna(axis=1, how='all')                 # 쓰지 않은 선택 지표 열은 뺌
en_judge_acc = round(100 * en_acc, 1)

print(f'■ 번역 후보 비교 ({len(mt_eval)}건). 참고: 원문(영어) 감성 정확도 {en_judge_acc}%')
display(mt_compare)
mt_compare.to_csv(OUT_DIR / 'mt_compare.csv', index=False, encoding='utf-8-sig')

# 리뷰별 감성 변화 표: 번역 후 감성 점수가 원문보다 얼마나 움직였는지 (+면 더 긍정적으로 번역)
mt_samples = mt_eval[['review_id', 'star', 'src_en']].rename(columns={'star': '별점', 'src_en': '원문'}).copy()
for name, ko in mt_out.items():
    mt_samples[name] = ko
    mt_samples[f'{name} 감성 변화'] = np.round(shifts[name], 2)
    if name in back:
        mt_samples[f'{name} 역번역'] = back[name]
mt_samples.to_csv(OUT_DIR / 'mt_samples.csv', index=False, encoding='utf-8-sig')

best = mt_compare.iloc[0]['번역 모델']
worst_idx = np.argsort(-np.abs(shifts[best]))[:5]
print(f'■ 1위({best}) 번역에서 감성이 가장 크게 바뀐 리뷰 5건 (전체는 mt_samples.csv)')
display(mt_samples.loc[worst_idx, ['별점', '원문', best, f'{best} 감성 변화']])

config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

[역번역 실패] OSError("Can't load the model for 'facebook/m2m100_1.2B'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/m2m100_1.2B' is the correct path to a directory containing a file named pytorch_
■ 번역 후보 비교 (300건). 참고: 원문(영어) 감성 정확도 73.5%


/tmp/ipykernel_581/2164981383.py:151: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mt_compare['종합 점수'] = (0.85 * mt_compare['감성 유지 점수'] + 0.15 * mt_compare['역번역 chrF'].fillna(0)).round(1)


,번역 모델,종류,라이선스,감성 유지 점수,감성 일치율(%),극성 반전율(%),강도 유지율(%),분포 유사도(%),LLM 별점 일치율(%),LLM 별점 차이,번역문 감성 정확도(%),실패율(%),실패 유형,속도(건/초),종합 점수
0,Gemini-3.5-FlashLite,gemini,상용 API,82.3,80.7,2.2,89.2,87.199997,89.7,0.10,70.7,0.3,한글 비율 낮음 1,1.9,70.0
1,NHNDQ-NLLB-en2ko,nllb,CC-BY-4.0(기반 NC),77.5,78.7,4.0,88.9,86.699997,87.7,0.13,69.2,1.7,"한글 비율 낮음 4, 반복 1",15.1,65.9
2,NLLB-1.3B,nllb,CC-BY-NC-4.0,77.4,78.0,2.5,87.9,85.500000,78.3,0.24,67.0,1.0,"반복 2, 길이 비정상 1",6.4,65.8
3,Gugugo-7B,llm,Apache-2.0,77.3,79.0,3.5,89.2,87.099998,82.0,0.19,70.3,0.3,한글 비율 낮음 1,0.3,65.7
4,NLLB-600M,nllb,CC-BY-NC-4.0,74.2,75.7,3.7,87.2,84.500000,78.7,0.24,65.5,1.7,"한글 비율 낮음 4, 반복 1",11.0,63.1
5,M2M100-1.2B,m2m,MIT,74.0,76.3,3.5,87.2,84.000000,76.3,0.27,66.5,4.3,"반복 7, 한글 비율 낮음 5, 빈 번역 1",6.2,62.9
6,Qwen2.5-7B,llm,Apache-2.0,71.2,78.0,5.3,86.5,83.599998,79.3,0.24,70.5,6.0,"길이 비정상 10, 한글 비율 낮음 8",0.3,60.5
7,M2M100-418M,m2m,MIT,69.5,73.3,5.2,86.2,83.599998,76.3,0.27,64.5,3.0,"반복 5, 길이 비정상 2, 한글 비율 낮음 2",12.0,59.1
8,Rosetta-4B,llm,Gemma,43.3,16.2,0.0,71.0,65.900002,20.0,1.65,20.0,100.0,빈 번역 300,0.5,36.8


■ 1위(Gemini-3.5-FlashLite) 번역에서 감성이 가장 크게 바뀐 리뷰 5건 (전체는 mt_samples.csv)


,별점,원문,Gemini-3.5-FlashLite,Gemini-3.5-FlashLite 감성 변화
289,4,Way too big. Definitely runs large!,너무 커요. 확실히 크게 나왔네요!,1.14
243,4,opaque. soooooo opaque and dark which is good i guess but i wanted sheer. my fault,불투명함. 엄청나게 불투명하고 어두워서 뭐 좋다면 좋겠지만 난 시스루를 원했음. 내 잘못이지.,-1.10
57,2,"Too thick. Good socks if want thick socks, I wanted light weight. I found them on NB websight.","너무 두껍네요. 두꺼운 양말을 원하신다면 좋은 양말이겠지만, 저는 가벼운 걸 원했거든요. NB 웹사이트에서 찾았습니다.",-0.96
208,3,"Very pretty and soft. Great shirt, lots of compliments but after 2-3 washes it looks 2 years old already. Kept its shape but got very pilly. Makes it look old.","매우 예쁘고 부드러워요. 훌륭한 셔츠이고 칭찬도 많이 받았지만, 2~3번 세탁하고 나니 벌써 2년은 된 옷처럼 보입니다. 모양은 유지되는데 보풀이 엄청 심하게 생겼어요. 그래서 너무 낡아 보여요.",-0.77
133,2,Loose fitting. Shoulders were to loose,헐렁한 핏. 어깨 부분이 너무 헐렁했어요.,-0.77


## 7. 번역 모델 선택과 전체 번역
`MAIN_MT='auto'`면 위 표의 종합 점수 1위를 고릅니다(실패율 5% 초과 제외, `AUTO_ALLOW_API=False`면 Gemini·Papago 제외).
번역 LLM은 번역 전용 모델보다 훨씬 느립니다. 아래에 비교 때 잰 속도로 2만 건 번역 시간을 어림해 보여 주니, 너무 길면 `MAIN_MT`로 다른 후보를 고르세요.
직접 고르려면 1장 설정에서 `MAIN_MT='M2M100-1.2B'`처럼 이름을 넣으세요. 상용 서비스가 목표라면 비상업(NC) 라이선스 후보는 피하세요.

In [12]:
if MAIN_MT == 'auto':
    cand = mt_compare[mt_compare['실패율(%)'] <= 5]
    if not AUTO_ALLOW_API:
        cand = cand[cand['번역 모델'].map(lambda n: MT_BY_NAME[n]['kind'] not in ('gemini', 'papago'))]
    assert len(cand), '자동으로 고를 번역 후보가 없습니다. MAIN_MT를 직접 지정하세요.'
    main_name = cand.iloc[0]['번역 모델']
else:
    main_name = MAIN_MT
MAIN_CFG = MT_BY_NAME[main_name]
print(f'전체 번역 모델: {main_name} ({MAIN_CFG["license"]})')
if mt_speed.get(main_name):   # 비교 때 잰 속도로 남은 번역 시간 어림
    print(f'  비교 때 속도 {mt_speed[main_name]:.1f}건/초 → 2만 건 번역에 약 {len(df) / mt_speed[main_name] / 3600:.1f}시간 (캐시된 건은 제외)')

ko_all, n_new, secs = get_translations(MAIN_CFG, df[['review_id', 'src_en']], '(전체)')
df['ko_text'] = ko_all.values
ko_map = df.set_index('review_id')['ko_text']
for d in (train_df, val_df, test_df):
    d['ko_text'] = d['review_id'].map(ko_map).fillna('')

fail = df.apply(lambda r: failure_reason(r['src_en'], r['ko_text']), axis=1)
print(f'번역 실패로 보이는 리뷰: {(fail != "").sum():,}건 / {len(df):,}건')
display(fail[fail != ''].value_counts().to_frame('건수'))
display(test_df[['star', 'src_en', 'ko_text']].head(5))

전체 번역 모델: NHNDQ-NLLB-en2ko (CC-BY-4.0(기반 NC))
  비교 때 속도 15.1건/초 → 2만 건 번역에 약 0.4시간 (캐시된 건은 제외)
[NHNDQ-NLLB-en2ko] 번역 19,700건 (전체)


config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

NHNDQ-NLLB-en2ko:   0%|          | 0/26 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=278) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=232) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=236) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=222) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  저장 19,700/19,700
번역 실패로 보이는 리뷰: 224건 / 20,000건


,건수
한글 비율 낮음,167
반복,56
길이 비정상,1


,star,src_en,ko_text
0,4,Blue loop earrings. I like them just as described,블루 루프 귀걸이. 설명한 대로 마음에 듭니다.
1,5,Great quality I come to expect from carhartt. Really nice look. Well made. What I expect from carhartt. Will buy again for sure.,카하르트에서 기대하는 좋은 품질. 정말 멋진 외관. 잘 만들어졌습니다. 카하르트에서 기대하는 바. 확실히 다시 살 것입니다.
2,4,"Cute and fashionable. Great find but i had to return, my shoe size looks huge in these boots size 10W, just my opinion. also bought in black super super cut...","귀엽고 패셔너블합니다. 좋은 것을 찾았지만 돌아와야 했습니다. 제 신발 사이즈는 이 부츠 사이즈 10W에서 엄청나게 보입니다. 또한 검은색 슈퍼 슈퍼 귀여운 옷을 샀고, 우리 알래스카 여행에서 지속되었으며, 검은 얼음 위에서 여전히 미끄러웠지만 발을 건조하고 따뜻하게 유지하는 데..."
3,4,SIZE UP. I sized up just to be cautious & glad I made the decision to do so. besides the sleeves hugging my arms it’s manageable& soft,SIZE UP. 나는 조심스럽고 그렇게 결정한 것을 기쁘게 생각합니다. 팔을 안고 있는 소매 외에도 그것은 관리 가능하고 부드럽습니다.
4,5,Shoe. My granddaughter loves playing dress up this completion of her outfit,신발. 내 손녀는 의상을 완성하기 위해 드레스 플레이를 좋아합니다.


## 8. 한국어 감성 모델 학습·평가
모델마다 결과를 `results/모델이름.json`에 저장합니다. 번역 모델을 바꾸면 결과도 달라지므로, 파일에 번역 모델 이름을 같이 적고 다르면 다시 실행합니다.

In [13]:
def run_model(cfg):
    """(가능하면) 학습 전 평가 → 파인튜닝 → 결과 묶기. 입력은 TEXT_COL(한국어 번역문)."""
    texts = test_df[TEXT_COL].tolist()
    print(f"\n===== {cfg['short']} ({cfg['hf']}) =====")
    zs_labels = zero_shot(cfg, texts) if cfg.get('zero_shot', True) else None
    ft = fine_tune(cfg)
    return dict(short=cfg['short'], hf=cfg['hf'], base=cfg['base'], license=cfg['license'],
                mt=main_name, review_ids=test_df['review_id'].tolist(), zs_labels=zs_labels, **ft)


ko_results, failed = {}, {}
for cfg in MODELS:
    path = RESULTS_DIR / f"{cfg['short']}.json"
    if RESUME and path.exists():
        r = json.loads(path.read_text())
        if r.get('review_ids') == test_df['review_id'].tolist() and r.get('mt') == main_name:
            ko_results[cfg['short']] = r
            print(f"[건너뜀] {cfg['short']}: 저장된 결과 사용")
            continue
    try:
        r = run_model(cfg)
    except Exception as e:
        import traceback
        failed[cfg['short']] = traceback.format_exc()[-1500:]   # 원인을 파일로 남김
        print(f"[실패] {cfg['short']}: {repr(e)[:300]}")
        free_memory()
        continue
    path.write_text(json.dumps(r, ensure_ascii=False))
    ko_results[cfg['short']] = r

(OUT_DIR / 'failed_models.json').write_text(json.dumps(failed, ensure_ascii=False, indent=1))
print('\n완료:', list(ko_results), '| 실패:', list(failed) or '없음')


===== KcELECTRA (beomi/KcELECTRA-base) =====


config.json:   0%|          | 0.00/514 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

  epoch 1: train_loss=1.1179  val_acc=0.5600  val_macroF1=0.5630


clapAI-XLMR epoch 2:   0%|          | 0/438 [00:00<?, ?it/s]

  epoch 2: train_loss=0.9204  val_acc=0.5683  val_macroF1=0.5669


clapAI-XLMR epoch 3:   0%|          | 0/438 [00:00<?, ?it/s]

  epoch 3: train_loss=0.8219  val_acc=0.5643  val_macroF1=0.5656
  검증 성능이 나아지지 않아 조기 종료합니다.

완료: ['KcELECTRA', 'KLUE-RoBERTa', 'nlptown-mBERT', 'tabularisai-DistilBERT', 'clapAI-XLMR'] | 실패: 없음


## 9. 영어 원문 vs 한국어 번역 성능
같은 테스트 리뷰에서 한국어 모델 결과(`[KO]`)와 이전 영어 결과(`[EN]`)를 한 표에 놓습니다.

In [14]:
def pct(x):
    return None if x is None or (isinstance(x, float) and np.isnan(x)) else round(100 * x, 1)


results, metrics = {}, {}
for lang, res in (('KO', ko_results), ('EN', en_results)):
    for short, r in res.items():
        name = f'{short} [{lang}]'
        r = dict(r, lang=lang)
        m = compute_metrics(test_df['star'], r['test_pred'], test_df['category'])
        if r.get('zs_labels'):
            m.update(zero_shot_metrics(r['zs_labels'], test_df['star']))
        results[name], metrics[name] = r, m

summary_df = pd.DataFrame([{
    '모델': name, '언어': results[name]['lang'], 'HF 모델': results[name]['hf'], '기반': results[name]['base'],
    '라이선스': results[name]['license'], '파라미터(M)': round(results[name]['n_params'] / 1e6),
    '정확도(%)': pct(m['accuracy']), 'Macro-F1(%)': pct(m['macro_f1']), '±1점 정확도(%)': pct(m['within1']),
    'MAE': round(m['mae'], 3), '감성 정확도(%)': pct(m['sent_accuracy']),
    '학습 전 감성 정확도(%)': pct(m.get('zs_sent_accuracy')), '학습 전 별점 정확도(%)': pct(m.get('zs_star_accuracy')),
    '최적 에폭': results[name]['best_epoch'], '학습 시간(분)': round(results[name]['train_minutes'], 1),
    '예측 속도(건/초)': round(results[name]['infer_per_sec']),
} for name, m in metrics.items()]).sort_values('정확도(%)', ascending=False).reset_index(drop=True)

per_star_df = pd.DataFrame({n: {f'{s}점': pct(m['per_star'].get(s)) for s in STARS} for n, m in metrics.items()}).T
per_cat_df  = pd.DataFrame({n: {c: pct(v) for c, v in m['per_category'].items()} for n, m in metrics.items()}).T
per_sent_df = pd.DataFrame({n: {s: pct(m['per_sentiment'].get(s)) for s in SENT_ORDER} for n, m in metrics.items()}).T

# 같은 모델의 영어 → 한국어 변화 (%p)
delta_rows = []
for short in sorted(set(ko_results) & set(en_results)):
    k, e = metrics[f'{short} [KO]'], metrics[f'{short} [EN]']
    delta_rows.append({'모델': short,
                       '정확도 EN': pct(e['accuracy']), '정확도 KO': pct(k['accuracy']),
                       '정확도 변화(%p)': round(pct(k['accuracy']) - pct(e['accuracy']), 1),
                       '감성 EN': pct(e['sent_accuracy']), '감성 KO': pct(k['sent_accuracy']),
                       '감성 변화(%p)': round(pct(k['sent_accuracy']) - pct(e['sent_accuracy']), 1)})
delta_df = pd.DataFrame(delta_rows)


def show(df_, higher_is_better=True, cols=None):
    try:
        st = df_.style.format(precision=1, na_rep='-')
        cols = cols or df_.select_dtypes('number').columns
        st = (st.highlight_max if higher_is_better else st.highlight_min)(subset=cols, props='font-weight:bold;background-color:#cde2fb')
        display(st)
    except Exception:
        display(df_)


print(f'번역 모델: {main_name}')
print('■ 모델 요약 (KO = 한국어 번역문, EN = 영어 원문)'); show(summary_df, cols=['정확도(%)', 'Macro-F1(%)', '±1점 정확도(%)', '감성 정확도(%)'])
if len(delta_df):
    print('■ 같은 모델의 영어 → 한국어 변화'); display(delta_df)
print('■ 별점별 정확도(%)'); show(per_star_df)
print('■ 카테고리별 정확도(%)'); show(per_cat_df)
print('■ 감성별 정확도(%)'); show(per_sent_df)

summary_df.to_csv(OUT_DIR / 'metrics_summary.csv', index=False, encoding='utf-8-sig')
delta_df.to_csv(OUT_DIR / 'metrics_en_vs_ko.csv', index=False, encoding='utf-8-sig')
pd.concat({'별점별': per_star_df, '카테고리별': per_cat_df, '감성별': per_sent_df}, axis=1).to_csv(
    OUT_DIR / 'metrics_by_item.csv', encoding='utf-8-sig')
(OUT_DIR / 'metrics_full.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=1))

번역 모델: NHNDQ-NLLB-en2ko
■ 모델 요약 (KO = 한국어 번역문, EN = 영어 원문)


,모델,언어,HF 모델,기반,라이선스,파라미터(M),정확도(%),Macro-F1(%),±1점 정확도(%),MAE,감성 정확도(%),학습 전 감성 정확도(%),학습 전 별점 정확도(%),최적 에폭,학습 시간(분),예측 속도(건/초)
0,nlptown-mBERT [EN],EN,nlptown/bert-base-multilingual-uncased-sentiment,mBERT,MIT,167,58.1,57.8,94.5,0.5,76.5,75.8,56.4,2,6.8,334
1,clapAI-XLMR [KO],KO,clapAI/roberta-base-multilingual-sentiment,XLM-R,Apache-2.0,278,56.9,56.7,93.1,0.5,75.9,70.8,35.0,2,7.5,474
2,KLUE-RoBERTa [KO],KO,klue/roberta-base,RoBERTa(한국어),카드 미표기,111,56.0,55.9,93.2,0.5,75.1,-,-,3,5.9,472
3,cardiff-XLMR [EN],EN,cardiffnlp/twitter-xlm-roberta-base-sentiment,XLM-R,미표기,278,55.5,55.7,93.5,0.5,74.7,69.1,32.3,1,5.3,404
4,tabularisai-DistilBERT [EN],EN,tabularisai/multilingual-sentiment-analysis,DistilBERT,CC-BY-NC-4.0,135,53.9,53.9,92.9,0.5,73.7,65.8,36.8,2,3.9,780
5,nlptown-mBERT [KO],KO,nlptown/bert-base-multilingual-uncased-sentiment,mBERT,MIT,167,53.5,53.6,92.2,0.6,73.1,61.5,41.4,3,6.7,401
6,KcELECTRA [KO],KO,beomi/KcELECTRA-base,ELECTRA(한국어),MIT,109,53.0,52.8,91.9,0.6,72.6,-,-,3,5.6,451
7,lxyuan-DistilBERT [EN],EN,lxyuan/distilbert-base-multilingual-cased-sentiments-student,DistilBERT,Apache-2.0,135,52.9,52.7,91.9,0.6,73.0,62.6,30.3,3,3.9,783
8,tabularisai-DistilBERT [KO],KO,tabularisai/multilingual-sentiment-analysis,DistilBERT,CC-BY-NC-4.0,135,51.7,51.5,90.9,0.6,72.3,61.9,36.1,3,3.8,812


■ 같은 모델의 영어 → 한국어 변화


,모델,정확도 EN,정확도 KO,정확도 변화(%p),감성 EN,감성 KO,감성 변화(%p)
0,nlptown-mBERT,58.1,53.5,-4.6,76.5,73.1,-3.4
1,tabularisai-DistilBERT,53.9,51.7,-2.2,73.7,72.3,-1.4


■ 별점별 정확도(%)


...

■ 카테고리별 정확도(%)


,Clothing·Shoes,Fashion
KcELECTRA [KO],50.5,55.6
KLUE-RoBERTa [KO],55.7,56.2
nlptown-mBERT [KO],52.8,54.2
tabularisai-DistilBERT [KO],51.1,52.3
clapAI-XLMR [KO],56.8,57.0
cardiff-XLMR [EN],54.5,56.6
lxyuan-DistilBERT [EN],52.2,53.6
nlptown-mBERT [EN],57.9,58.3
tabularisai-DistilBERT [EN],52.9,55.0


■ 감성별 정확도(%)


,부정,중립,긍정
KcELECTRA [KO],75.9,41.3,85.0
KLUE-RoBERTa [KO],79.4,43.0,86.8
nlptown-mBERT [KO],74.9,45.0,85.5
tabularisai-DistilBERT [KO],74.7,41.0,85.7
clapAI-XLMR [KO],80.0,42.0,88.8
cardiff-XLMR [EN],80.5,40.7,86.0
lxyuan-DistilBERT [EN],78.9,39.5,83.8
nlptown-mBERT [EN],84.3,37.2,88.4
tabularisai-DistilBERT [EN],80.9,38.0,84.4


7839

## 10. 원래값 / 예측값 표
테스트 리뷰마다 영어 원문, 한국어 번역문, 실제 별점과 모델별 예측을 나란히 놓습니다. 전체는 `predictions_test.csv`.

In [15]:
pred_table = test_df[['review_id', 'category', 'date', 'star', 'sentiment', 'src_en', 'ko_text']].rename(
    columns={'category': '카테고리', 'date': '작성일', 'star': '원래 별점', 'sentiment': '원래 감성',
             'src_en': '영어 원문', 'ko_text': '한국어 번역'})
for name, r in results.items():
    pred = np.array(r['test_pred'])
    pred_table[f'{name} 예측'] = pred
    pred_table[f'{name} 확신도'] = r['test_conf']
    pred_table[f'{name} 정답'] = np.where(pred == pred_table['원래 별점'].values, 'O', 'X')

# 영어로는 맞혔는데 한국어로는 틀린 리뷰 (번역 때문에 생긴 오류 후보)
for short in sorted(set(ko_results) & set(en_results)):
    pred_table[f'{short} 번역 후 오답'] = ((pred_table[f'{short} [EN] 정답'] == 'O') &
                                         (pred_table[f'{short} [KO] 정답'] == 'X'))
pred_table.to_csv(OUT_DIR / 'predictions_test.csv', index=False, encoding='utf-8-sig')

print('■ 앞부분 10건'); display(pred_table.head(10))
flag_cols = [c for c in pred_table if c.endswith('번역 후 오답')]
if flag_cols:
    print('■ 영어로는 맞혔지만 한국어 번역으로는 틀린 리뷰 10건 (번역 오류 후보)')
    display(pred_table[pred_table[flag_cols[0]]][['원래 별점', '영어 원문', '한국어 번역']].head(10))

■ 앞부분 10건


,review_id,카테고리,작성일,원래 별점,원래 감성,영어 원문,한국어 번역,KcELECTRA [KO] 예측,KcELECTRA [KO] 확신도,KcELECTRA [KO] 정답,...,lxyuan-DistilBERT [EN] 확신도,lxyuan-DistilBERT [EN] 정답,nlptown-mBERT [EN] 예측,nlptown-mBERT [EN] 확신도,nlptown-mBERT [EN] 정답,tabularisai-DistilBERT [EN] 예측,tabularisai-DistilBERT [EN] 확신도,tabularisai-DistilBERT [EN] 정답,nlptown-mBERT 번역 후 오답,tabularisai-DistilBERT 번역 후 오답
0,2540ad0ef94f,Clothing·Shoes,2021-04-14,4,긍정,Blue loop earrings. I like them just as described,블루 루프 귀걸이. 설명한 대로 마음에 듭니다.,5,0.5775,X,...,0.5996,X,5,0.5640,X,4,0.4828,O,False,True
1,de105c1a6235,Clothing·Shoes,2020-11-21,5,긍정,Great quality I come to expect from carhartt. Really nice look. Well made. What I expect from carhartt. Will buy again for sure.,카하르트에서 기대하는 좋은 품질. 정말 멋진 외관. 잘 만들어졌습니다. 카하르트에서 기대하는 바. 확실히 다시 살 것입니다.,5,0.8823,O,...,0.8835,O,5,0.8133,O,5,0.8282,O,False,False
2,e4454665e4a4,Clothing·Shoes,2022-11-16,4,긍정,"Cute and fashionable. Great find but i had to return, my shoe size looks huge in these boots size 10W, just my opinion. also bought in black super super cut...","귀엽고 패셔너블합니다. 좋은 것을 찾았지만 돌아와야 했습니다. 제 신발 사이즈는 이 부츠 사이즈 10W에서 엄청나게 보입니다. 또한 검은색 슈퍼 슈퍼 귀여운 옷을 샀고, 우리 알래스카 여행에서 지속되었으며, 검은 얼음 위에서 여전히 미끄러웠지만 발을 건조하고 따뜻하게 유지하는 데...",5,0.5562,X,...,0.6013,X,4,0.6459,O,4,0.6059,O,False,False
3,1eb401b30fd6,Fashion,2022-09-14,4,긍정,SIZE UP. I sized up just to be cautious & glad I made the decision to do so. besides the sleeves hugging my arms it’s manageable& soft,SIZE UP. 나는 조심스럽고 그렇게 결정한 것을 기쁘게 생각합니다. 팔을 안고 있는 소매 외에도 그것은 관리 가능하고 부드럽습니다.,4,0.4862,O,...,0.4041,X,4,0.6109,O,5,0.6049,X,True,False
4,3e6b1771df58,Fashion,2022-06-01,5,긍정,Shoe. My granddaughter loves playing dress up this completion of her outfit,신발. 내 손녀는 의상을 완성하기 위해 드레스 플레이를 좋아합니다.,5,0.8627,O,...,0.6911,O,5,0.7088,O,5,0.8205,O,False,False
5,0bf955388404,Clothing·Shoes,2021-01-10,4,긍정,Nice dress. I would disagree this dress being a maxi dress. Midi at the most. It's marketed as toddler dress but comes up to size 10-11. I got it for my 10 ...,멋진 드레스입니다. 이 드레스는 맥시 드레스라는 데 동의하지 않을 것입니다. 최소 미디입니다. 유아용 드레스로 판매되지만 10-11 사이즈까지 나온다. 10살짜리용으로 구입했습니다. 무릎 아래로 약간 들어갑니다. 가슴 높이에는 뒷면에 2개의 넥타이가 있습니다. 목 뒤에는 지퍼가 ...,4,0.7276,O,...,0.7608,O,4,0.8392,O,4,0.7763,O,False,False
6,ced1887d30ae,Fashion,2021-11-30,1,부정,Very Too Narrow for a XXL. Look at the picture. The yellow sock is truly an XXL. The Blue socks are way to narrow and they don't stretch enough for my wide ...,XXL에는 너무 좁다. 사진을 봐. 노란색 양말은 정말 XXL입니다. 파란색 양말은 너무 좁고 넓은 발에 맞지 않습니다. 넓은 양말이 필요합니까? 이것을 피하십시오.,1,0.4991,O,...,0.5110,X,2,0.4861,X,2,0.4906,X,False,False
7,a2eddfd77e15,Clothing·Shoes,2020-12-04,1,부정,"uncomfortable. I really wanted makes made in US, but these, yikes! So uncomfortable. Fits flat to face, really can't breath with these on. Not a good purchase.","불편해. 미국에서 만든 것을 정말 원했지만, 이것들은, ! 너무 불편해. 얼굴에 평평하게 맞아서 정말 숨을 쉴 수 없습니다. 좋은 구매가 아닙니다.",1,0.6078,O,...,0.4821,O,1,0.6971,O,2,0.4431,X,False,False
8,96818f0eca24,Clothing·Shoes,2021-11-16,2,부정,Falls apart. These do have the great Eagle Eyes lenses but they are too large and I am constantly chasing parts that keep flying off the night vision lenses...,떨어집니다. 이 제품에는 훌륭한 이글 아이즈 렌즈가 있지만 너무 커서 야간 투시경 렌즈에서 계속 날아가는 부품을 계속 쫓고 있습니다. 이 이중 기능 안경의 엔지니어링을 개선해야 합니다.,2,0.4616,O,...,0.4563,O,2,0.5475,O,2,0.4594,O,False,False
9,2f91201c0ee2,Clothing·Shoes,2020-11-04,2,부정,Cheap. The material is SUPER CHEAP ! If you want this to like throw on over a swim suit or something I’d say it would work but to just wear out and about ...no,저렴합니다. 재료는 매우 저렴합니다! 수영복이나 그 위에 던지는 것을 좋아하고 싶다면 나는 그것이 효과가 있다고 말하고 단지 마모되고 거의 ... 아니요.,1,0.5489,X,...,0.7302,X,1,0.5945,X,1,0.7565,X,False,False


■ 영어로는 맞혔지만 한국어 번역으로는 틀린 리뷰 10건 (번역 오류 후보)


,원래 별점,영어 원문,한국어 번역
3,4,SIZE UP. I sized up just to be cautious & glad I made the decision to do so. besides the sleeves hugging my arms it’s manageable& soft,SIZE UP. 나는 조심스럽고 그렇게 결정한 것을 기쁘게 생각합니다. 팔을 안고 있는 소매 외에도 그것은 관리 가능하고 부드럽습니다.
16,5,Make them last. These has saved me from a lot of my jewelry braking,오래 지속되도록 합니다. 이것으로 인해 장신구 제동이 많이 되지 않습니다.
30,5,Just what I was looking for. Just what I was looking for and color assortment,딱 찾던 거 딱 찾던 거랑 색상 구색
38,4,Good.. Very cozy. Comfortable and medium weight. Nice for the between weather seasons.,좋아요.. 매우 아늑합니다. 편안하고 중간 무게입니다. 날씨 사이에는 좋습니다.
56,4,Missing ball on one ring. Came quick but was missing one of the balls for the black horseshoe. Also the spikes come off kinda easy (could be that i didn’t t...,한 링에 공이 빠졌다. 빨리 왔지만 검은 말굽을 위한 공 중 하나가 빠졌다. 또한 스파이크가 약간 쉽게 떨어져 나간다(아마도 충분히 조이지 않았을 수 있다). 그래서 공이 빠졌기 때문에 스타를 제거하여 사용할 수 없습니다:/
68,5,"Great quality. Iam a female. Im 5'1 -5'2 weigh 110. I got the size small, though fit me long an just a bit baggy. Expected since these are for men. Though t...",좋은 품질입니다. 암컷입니다. Im 5'1 -5'2 무게는 110입니다. 저는 크기가 작지만 길고 약간 뭉툭합니다. 남성용이기 때문에 예상됩니다. 품질은 훌륭하고 매우 부드럽습니다. 매우 얇고 운동에 좋습니다. 땀을 흘리지 않도록 도와줍니다. 디피네틀리는 다시 살 것입니다!
71,2,"If you're more than a b-cup - skip these. No support, which was expected, but I literally fall out of these over the course of the day. The straps offer no ...","만약 당신이 b-컵 이상이라면, 이것들을 건너다. 기대했던 지원은 없지만, 나는 말 그대로 하루 동안 이것들로부터 떨어진다. 스트랩은 전혀 지원을 제공하지 않는다."
76,1,"Wrongly labeled. pants were actually 4"" shorter than they were supposed to be.",라벨을 잘못 붙여서요. 바지는 원래보다 4인치 짧았습니다.
88,4,Sturdy for rugged work or gardening. I've used these gloves for years and found them to be the best value for the money in sturdiness and grip. The plastic ...,"견고한 작업이나 정원 가꾸기에 견고합니다. 나는 이 장갑을 몇 년 동안 사용했으며 견고함과 그립감에서 돈에 대한 최고의 가치가 있음을 발견했다. 플라스틱은 결국 스티어링 휠의 지속적인 마모로 인해 엄지손가락을 넘어 부서지고 마모되지만, 다른 곳을 찾을 이유가 충분하지 않습니다. ..."
96,2,Shrunk up to a belly shirt. Would order big for washing this shirt shrunk up to my daughters size.,뱃살 셔츠로 수축. 이 셔츠를 내 딸의 크기에 맞게 수축하여 세탁하기 위해 크게 주문할 것입니다.


## 11. 비교 화면 (대시보드)
이전 노트북의 비교 화면에 **번역 모델 비교 표**와 **번역문/원문 함께 보기**를 더했습니다. `dashboard.html`로 저장됩니다.

In [16]:
DASHBOARD_TEMPLATE = r"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>번역 리뷰 별점 비교</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/plotly.js/2.35.3/plotly.min.js"></script>
<style>
:root{color-scheme:light;--page:#f9f9f7;--surface:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--muted:#898781;
 --grid:#e1e0d9;--axis:#c3c2b7;--border:rgba(11,11,11,.10);--s1:#2a78d6;--s2:#eb6834;
 --good:#006300;--bad:#d03b3b;--hl:#cde2fb;}
@media (prefers-color-scheme:dark){:root:not([data-theme="light"]){color-scheme:dark;--page:#0d0d0d;--surface:#1a1a19;
 --ink:#ffffff;--ink2:#c3c2b7;--muted:#898781;--grid:#2c2c2a;--axis:#383835;--border:rgba(255,255,255,.10);
 --s1:#3987e5;--s2:#d95926;--good:#0ca30c;--bad:#e66767;--hl:#184f95;}}
:root[data-theme="dark"]{color-scheme:dark;--page:#0d0d0d;--surface:#1a1a19;--ink:#ffffff;--ink2:#c3c2b7;--muted:#898781;
 --grid:#2c2c2a;--axis:#383835;--border:rgba(255,255,255,.10);--s1:#3987e5;--s2:#d95926;--good:#0ca30c;--bad:#e66767;--hl:#184f95;}
*{box-sizing:border-box}
body{margin:0;background:var(--page);color:var(--ink);font:14px/1.5 system-ui,-apple-system,"Segoe UI","Apple SD Gothic Neo","Malgun Gothic",sans-serif}
main{max-width:1200px;margin:0 auto;padding:24px 16px 48px}
h1{font-size:22px;margin:0 0 4px}
h2{font-size:16px;margin:0 0 4px}
.sub{color:var(--ink2);margin:0 0 16px}
.lead{font-size:16px;margin:0 0 20px}
.card{background:var(--surface);border:1px solid var(--border);border-radius:10px;padding:16px;margin-bottom:16px;min-width:0}
.note{color:var(--ink2);font-size:12px;margin:4px 0 8px}
.grid2{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:16px}
@media (max-width:520px){.grid2{grid-template-columns:1fr}}
.chart{width:100%;height:340px}
.tablewrap{overflow-x:auto}
table{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}
th,td{padding:6px 8px;border-bottom:1px solid var(--grid);text-align:right;white-space:nowrap}
th{color:var(--ink2);font-weight:600;position:sticky;top:0;background:var(--surface)}
td.l,th.l{text-align:left}
td.best{font-weight:700;background:var(--hl)}
.controls{display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin:8px 0 12px}
.controls label{color:var(--ink2);font-size:13px}
select,input[type=search]{font:inherit;color:var(--ink);background:var(--surface);border:1px solid var(--axis);border-radius:6px;padding:4px 8px}
.seg button{font:inherit;color:var(--ink2);background:transparent;border:1px solid var(--axis);padding:4px 10px;cursor:pointer}
.seg button:first-child{border-radius:6px 0 0 6px}.seg button:last-child{border-radius:0 6px 6px 0}
.seg button+button{border-left:none}
.seg button[aria-pressed=true]{color:var(--ink);background:var(--hl);font-weight:600}
.ok{color:var(--good);font-weight:600}.ng{color:var(--bad);font-weight:600}
.txt{min-width:320px;max-width:520px;white-space:normal;text-align:left;color:var(--ink2)}
.pred{height:520px;overflow:auto}
</style></head><body><main>
<h1>영어 원문 vs 한국어 번역: 별점 예측 모델 비교</h1>
<p class="sub" id="meta"></p>
<p class="lead" id="lead"></p>

<section class="card"><h2>번역 모델 비교</h2>
<p class="note" id="mtNote"></p>
<div class="tablewrap"><table id="mtTable"></table></div></section>

<section class="card"><h2>모델 요약</h2>
<p class="note">테스트셋 기준. 열마다 가장 좋은 값을 강조했습니다. MAE는 작을수록 좋습니다.</p>
<div class="tablewrap"><table id="summary"></table></div></section>

<div class="grid2">
<section class="card"><h2>모델별 지표</h2>
<div class="controls seg" id="metricBtns" role="group" aria-label="지표 선택"></div>
<div id="barChart" class="chart"></div></section>
<section class="card"><h2>학습 전후 감성 정확도</h2>
<p class="note">부정·중립·긍정 3분류 기준. 긍정/부정만 내는 모델은 학습 전에 중립을 맞힐 수 없습니다.</p>
<div id="zsChart" class="chart"></div></section>
</div>

<div class="grid2">
<section class="card"><h2>별점별 정확도</h2>
<p class="note">실제 그 별점인 리뷰 중 정확히 맞힌 비율(%)</p>
<div id="starHeat" class="chart"></div></section>
<section class="card"><h2>카테고리별 정확도</h2>
<p class="note">카테고리 안에서 별점을 정확히 맞힌 비율(%)</p>
<div id="catHeat" class="chart"></div></section>
</div>

<section class="card"><h2>혼동행렬</h2>
<div class="controls"><label for="cmModel">모델</label><select id="cmModel"></select>
<span class="note">행 = 실제 별점, 열 = 예측 별점. 칸의 값은 실제 별점 기준 비율(%)이고, 마우스를 올리면 건수가 보입니다.</span></div>
<div id="cmChart" class="chart" style="height:380px;max-width:560px"></div></section>

<section class="card"><h2>원래값 / 예측값</h2>
<div class="controls">
<label for="fCat">카테고리</label><select id="fCat"></select>
<label for="fStar">실제 별점</label><select id="fStar"></select>
<label for="fModel">오답 기준 모델</label><select id="fModel"></select>
<label><input type="checkbox" id="fWrong"> 오답만</label>
<label><input type="checkbox" id="fSplit"> 모델끼리 예측이 갈린 리뷰만</label>
<input type="search" id="fText" placeholder="번역문·원문 검색" aria-label="본문 검색">
<span class="note" id="rowCount"></span></div>
<div class="tablewrap pred"><table id="preds"></table></div>
<p class="note" id="rowsNote"></p></section>
</main>
<script>
const DATA = __DATA__;
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const dark = () => css('--surface') === '#1a1a19';
const fmt = v => (v === null || v === undefined) ? '-' : v.toFixed(1);
const esc = s => String(s ?? '').replace(/[&<>"]/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
const models = DATA.models;

// 공통 차트 레이아웃: 얇은 격자, 축 색, 표면색을 CSS 변수에서 읽음 (다크 모드 대응)
function baseLayout(extra) {
  const ax = {gridcolor: css('--grid'), linecolor: css('--axis'), zerolinecolor: css('--axis'),
              tickfont: {color: css('--muted')}, title: {font: {color: css('--ink2')}}, automargin: true};
  return Object.assign({paper_bgcolor: css('--surface'), plot_bgcolor: css('--surface'),
    font: {color: css('--ink'), family: 'system-ui, -apple-system, "Segoe UI", sans-serif', size: 12},
    margin: {l: 10, r: 10, t: 10, b: 10}, xaxis: Object.assign({}, ax), yaxis: Object.assign({}, ax),
    hoverlabel: {bgcolor: css('--surface'), bordercolor: css('--axis'), font: {color: css('--ink')}},
    barcornerradius: 4, showlegend: false}, extra || {});
}
const cfg = {displayModeBar: false, responsive: true};
// 순차 색상(파랑 한 가지, 밝음→진함). 다크 모드에서는 낮은 값이 배경에 가깝게.
const seqScale = () => dark()
  ? [[0, '#1e2a3a'], [0.5, '#1c5cab'], [1, '#3987e5']]
  : [[0, '#eef5fd'], [0.5, '#5598e7'], [1, '#104281']];

// ── 헤더 ──
document.getElementById('meta').textContent =
  `기간 ${DATA.meta.period} · 카테고리 ${DATA.meta.categories.join(', ')} · 학습 ${DATA.meta.n_train.toLocaleString()} / 검증 ${DATA.meta.n_val.toLocaleString()} / 테스트 ${DATA.meta.n_test.toLocaleString()}건 · 생성 ${DATA.meta.generated}`;
if (models.length) {
  const b = models[0];   // 요약은 정확도 순으로 정렬되어 들어옴
  document.getElementById('lead').innerHTML =
    `가장 정확한 모델은 <b>${esc(b.name)}</b>입니다. 별점 정확도 ${fmt(b.accuracy)}%, ±1점 이내 ${fmt(b.within1)}%, 감성 3분류 ${fmt(b.sent_accuracy)}%.`;
} else {
  document.getElementById('lead').textContent = '완료된 모델 결과가 없습니다.';
}

// ── 요약 표 ──
const COLS = [
  ['name', '모델', 'l'], ['base', '기반', 'l'], ['params_m', '파라미터(M)'],
  ['accuracy', '정확도(%)', '', 'max'], ['macro_f1', 'Macro-F1(%)', '', 'max'], ['within1', '±1점 정확도(%)', '', 'max'],
  ['mae', 'MAE', '', 'min'], ['sent_accuracy', '감성 정확도(%)', '', 'max'],
  ['zs_sent_accuracy', '학습 전 감성(%)', '', 'max'], ['best_epoch', '최적 에폭'],
  ['train_minutes', '학습 시간(분)', '', 'min'], ['infer_per_sec', '예측 속도(건/초)', '', 'max'], ['license', '라이선스', 'l']];
(function () {
  const best = {};
  COLS.forEach(([k, , , dir]) => { if (!dir) return;
    const vals = models.map(m => m[k]).filter(v => v !== null && v !== undefined);
    best[k] = dir === 'max' ? Math.max(...vals) : Math.min(...vals); });
  let h = '<thead><tr>' + COLS.map(([, t, c]) => `<th class="${c || ''}">${t}</th>`).join('') + '</tr></thead><tbody>';
  models.forEach(m => {
    h += '<tr>' + COLS.map(([k, , c, dir]) => {
      let v = m[k];
      const isBest = dir && v === best[k] && models.length > 1;
      if (typeof v === 'number') v = (k === 'mae') ? v.toFixed(3) : (k === 'params_m' || k === 'infer_per_sec' || k === 'best_epoch') ? Math.round(v).toLocaleString() : fmt(v);
      return `<td class="${c || ''}${isBest ? ' best' : ''}">${isBest ? '▲ ' : ''}${esc(v ?? '-')}</td>`;
    }).join('') + '</tr>';
  });
  document.getElementById('summary').innerHTML = h + '</tbody>';
})();

// ── 지표별 막대 차트 (지표 선택 버튼) ──
const METRICS = [['accuracy', '정확도'], ['macro_f1', 'Macro-F1'], ['within1', '±1점 정확도'], ['sent_accuracy', '감성 정확도']];
function drawBar(key) {
  const label = METRICS.find(m => m[0] === key)[1];
  const ms = [...models].sort((a, b) => (a[key] ?? 0) - (b[key] ?? 0));   // 아래→위로 커지게
  Plotly.react('barChart', [{type: 'bar', orientation: 'h', x: ms.map(m => m[key]), y: ms.map(m => m.name),
    marker: {color: css('--s1')}, width: 0.55,
    text: ms.map(m => fmt(m[key]) + '%'), textposition: 'outside', textfont: {color: css('--ink2')}, cliponaxis: false,
    hovertemplate: '%{y}<br>' + label + ' %{x:.1f}%<extra></extra>'}],
    baseLayout({xaxis: Object.assign(baseLayout().xaxis, {range: [0, 105], ticksuffix: '%'}), margin: {l: 10, r: 40, t: 10, b: 10}}), cfg);
  document.querySelectorAll('#metricBtns button').forEach(b => b.setAttribute('aria-pressed', b.dataset.k === key));
}
METRICS.forEach(([k, t]) => {
  const b = document.createElement('button'); b.textContent = t; b.dataset.k = k;
  b.onclick = () => drawBar(k); document.getElementById('metricBtns').appendChild(b);
});
if (models.length) drawBar('accuracy');

// ── 학습 전후 비교 (주황 = 학습 전, 파랑 = 학습 후) ──
if (models.length) Plotly.react('zsChart', [
  {type: 'bar', name: '학습 전(제로샷)', x: models.map(m => m.name), y: models.map(m => m.zs_sent_accuracy),
   marker: {color: css('--s2'), line: {color: css('--surface'), width: 2}},
   hovertemplate: '%{x}<br>학습 전 %{y:.1f}%<extra></extra>'},
  {type: 'bar', name: '학습 후(파인튜닝)', x: models.map(m => m.name), y: models.map(m => m.sent_accuracy),
   marker: {color: css('--s1'), line: {color: css('--surface'), width: 2}},
   text: models.map(m => fmt(m.sent_accuracy) + '%'), textposition: 'outside', textfont: {color: css('--ink2')}, cliponaxis: false,
   hovertemplate: '%{x}<br>학습 후 %{y:.1f}%<extra></extra>'}],
  baseLayout({barmode: 'group', bargap: 0.35, bargroupgap: 0.08, showlegend: true,
    legend: {orientation: 'h', y: 1.12, x: 0, font: {color: css('--ink2')}},
    yaxis: Object.assign(baseLayout().yaxis, {range: [0, 108], ticksuffix: '%'}), margin: {l: 10, r: 10, t: 30, b: 10}}), cfg);

// ── 히트맵 공통 함수 ──
function heat(id, cols, getter, hoverName) {
  const z = models.map(m => cols.map(c => getter(m, c)));
  Plotly.react(id, [{type: 'heatmap', z: z, x: cols, y: models.map(m => m.name), zmin: 0, zmax: 100,
    colorscale: seqScale(), xgap: 2, ygap: 2, texttemplate: '%{z:.1f}', showscale: true,
    colorbar: {ticksuffix: '%', thickness: 10, outlinewidth: 0, tickfont: {color: css('--muted')}},
    hovertemplate: '%{y}<br>' + hoverName + ' %{x}: %{z:.1f}%<extra></extra>'}],
    baseLayout({yaxis: Object.assign(baseLayout().yaxis, {autorange: 'reversed'}), xaxis: Object.assign(baseLayout().xaxis, {side: 'top'})}), cfg);
}
if (models.length) {
  heat('starHeat', ['1점', '2점', '3점', '4점', '5점'], (m, c) => m.per_star[c], '실제');
  heat('catHeat', DATA.meta.categories, (m, c) => m.per_category[c], '');
}

// ── 혼동행렬 ──
const cmSel = document.getElementById('cmModel');
models.forEach(m => cmSel.add(new Option(m.name, m.name)));
function drawCM() {
  const m = models.find(x => x.name === cmSel.value); if (!m) return;
  const cm = m.confusion, rowsum = cm.map(r => r.reduce((a, b) => a + b, 0) || 1);
  const z = cm.map((r, i) => r.map(v => 100 * v / rowsum[i]));
  const stars = ['1점', '2점', '3점', '4점', '5점'];
  Plotly.react('cmChart', [{type: 'heatmap', z: z, x: stars, y: stars, customdata: cm, zmin: 0, zmax: 100,
    colorscale: seqScale(), xgap: 2, ygap: 2, texttemplate: '%{z:.0f}', showscale: false,
    hovertemplate: '실제 %{y} → 예측 %{x}<br>%{customdata}건 (%{z:.1f}%)<extra></extra>'}],
    baseLayout({xaxis: Object.assign(baseLayout().xaxis, {title: {text: '예측 별점'}, side: 'top'}),
                yaxis: Object.assign(baseLayout().yaxis, {title: {text: '실제 별점'}, autorange: 'reversed'})}), cfg);
}
cmSel.onchange = drawCM; drawCM();

// ── 번역 모델 비교 표 ──
(function () {
  const mt = DATA.mt || [];
  document.getElementById('mtNote').textContent = DATA.meta.mt_note || '';
  if (!mt.length) { document.getElementById('mtTable').innerHTML = '<tbody><tr><td class="l">번역 비교 결과가 없습니다.</td></tr></tbody>'; return; }
  const cols = Object.keys(mt[0]);
  const higher = {'감성 유지 점수': 1, '감성 일치율(%)': 1, '강도 유지율(%)': 1, '분포 유사도(%)': 1, 'LLM 별점 일치율(%)': 1,
    '번역문 감성 정확도(%)': 1, '역번역 chrF': 1, 'CometKiwi': 1, '속도(건/초)': 1, '종합 점수': 1,
    '극성 반전율(%)': -1, 'LLM 별점 차이': -1, '실패율(%)': -1};
  const best = {};
  cols.forEach(c => { if (!higher[c]) return; const v = mt.map(r => r[c]).filter(x => x !== null && x !== undefined);
    if (v.length) best[c] = higher[c] > 0 ? Math.max(...v) : Math.min(...v); });
  let h = '<thead><tr>' + cols.map(c => `<th class="${typeof mt[0][c] === 'string' ? 'l' : ''}">${esc(c)}</th>`).join('') + '</tr></thead><tbody>';
  mt.forEach(r => { const main = r['번역 모델'] === DATA.meta.main_mt;
    h += '<tr>' + cols.map(c => { const v = r[c], isBest = best[c] !== undefined && v === best[c] && mt.length > 1;
      const txt = (c === '번역 모델' && main) ? `${esc(v)} (사용)` : esc(v ?? '-');
      return `<td class="${typeof v === 'string' ? 'l' : ''}${isBest ? ' best' : ''}">${isBest ? '▲ ' : ''}${txt}</td>`; }).join('') + '</tr>'; });
  document.getElementById('mtTable').innerHTML = h + '</tbody>';
})();

// ── 원래값/예측값 표 ──
const rows = DATA.rows, names = models.map(m => m.name);
const fCat = document.getElementById('fCat'), fStar = document.getElementById('fStar'), fModel = document.getElementById('fModel');
fCat.add(new Option('전체', '')); DATA.meta.categories.forEach(c => fCat.add(new Option(c, c)));
fStar.add(new Option('전체', '')); [1, 2, 3, 4, 5].forEach(s => fStar.add(new Option(s + '점', s)));
fModel.add(new Option('아무 모델이나', '')); names.forEach(n => fModel.add(new Option(n, n)));
function renderRows() {
  const q = document.getElementById('fText').value.trim().toLowerCase();
  const wrong = document.getElementById('fWrong').checked, split = document.getElementById('fSplit').checked;
  const list = rows.filter(r => {
    if (fCat.value && r.category !== fCat.value) return false;
    if (fStar.value && r.star !== +fStar.value) return false;
    if (wrong) { const ms = fModel.value ? [fModel.value] : names; if (!ms.some(n => r.pred[n] !== r.star)) return false; }
    if (split && new Set(names.map(n => r.pred[n])).size < 2) return false;
    if (q && !((r.text + ' ' + (r.en || '')).toLowerCase().includes(q))) return false;
    return true; });
  const shown = list.slice(0, 500);
  let h = '<thead><tr><th class="l">카테고리</th><th class="l">작성일</th><th>원래 별점</th>' +
    names.map(n => `<th>${esc(n)}</th>`).join('') + '<th class="l">리뷰</th></tr></thead><tbody>';
  shown.forEach(r => {
    h += `<tr><td class="l">${esc(r.category)}</td><td class="l">${r.date}</td><td><b>${r.star}</b></td>` +
      names.map(n => { const p = r.pred[n], ok = p === r.star;
        return `<td class="${ok ? 'ok' : 'ng'}" title="확신도 ${(100 * r.conf[n]).toFixed(0)}%">${ok ? '✓' : '✗'} ${p}</td>`; }).join('') +
      `<td class="txt"><span style="color:var(--ink)">${esc(r.text)}</span><br>${esc(r.en || '')}</td></tr>`; });
  document.getElementById('preds').innerHTML = h + '</tbody>';
  document.getElementById('rowCount').textContent = `${list.length.toLocaleString()}건 중 ${shown.length.toLocaleString()}건 표시`;
}
['fCat', 'fStar', 'fModel', 'fWrong', 'fSplit', 'fText'].forEach(id => document.getElementById(id).addEventListener('input', renderRows));
document.getElementById('rowsNote').textContent = DATA.meta.rows_note;
renderRows();
</script></body></html>"""


def build_dashboard():
    """지표, 번역 비교, 예측 결과를 JSON으로 묶어 HTML 템플릿에 넣는다."""
    model_rows = []
    for _, s in summary_df.iterrows():
        name, m, r = s['모델'], metrics[s['모델']], results[s['모델']]
        model_rows.append(dict(
            name=name, base=r['base'], license=r['license'], params_m=r['n_params'] / 1e6,
            accuracy=pct(m['accuracy']), macro_f1=pct(m['macro_f1']), within1=pct(m['within1']), mae=m['mae'],
            sent_accuracy=pct(m['sent_accuracy']), zs_sent_accuracy=pct(m.get('zs_sent_accuracy')),
            best_epoch=r['best_epoch'], train_minutes=round(r['train_minutes'], 1), infer_per_sec=r['infer_per_sec'],
            per_star={f'{k}점': pct(v) for k, v in m['per_star'].items()},
            per_category={k: pct(v) for k, v in m['per_category'].items()},
            confusion=m['confusion']))

    sub = test_df.head(DASHBOARD_MAX_ROWS)
    rows = []
    for i, t in sub.iterrows():
        rows.append(dict(category=t['category'], date=t['date'], star=int(t['star']),
                         text=t['ko_text'][:500], en=t['src_en'][:500],
                         pred={n: int(results[n]['test_pred'][i]) for n in results},
                         conf={n: float(results[n]['test_conf'][i]) for n in results}))

    mt_rows = json.loads(mt_compare.to_json(orient='records', force_ascii=False))
    meta = dict(period=f'{START_DATE} ~ {(pd.Timestamp(END_DATE) - pd.Timedelta(days=1)).strftime("%Y-%m-%d")}',
                categories=list(CATEGORIES), n_train=len(train_df), n_val=len(val_df), n_test=len(test_df),
                generated=dt.datetime.now().strftime('%Y-%m-%d %H:%M'), main_mt=main_name,
                mt_note=f'테스트 리뷰 {len(mt_eval)}건 기준. 감성 유지 점수 = (감성 일치율 + 강도 유지율 + 분포 유사도 + LLM 별점 일치율) 평균 − 2×극성 반전율, 종합 점수 = 0.85×감성 유지 점수 + 0.15×역번역 chrF. '
                        f'원문(영어) 감성 정확도는 {en_judge_acc}%입니다. 전체 번역에는 {main_name}을(를) 썼습니다.',
                rows_note=f'테스트 {len(test_df):,}건 중 {len(rows):,}건. 위는 한국어 번역, 아래 회색은 영어 원문입니다. '
                          '[KO]는 번역문으로, [EN]은 원문으로 학습한 결과입니다. 전체는 predictions_test.csv에 있습니다.')
    payload = json.dumps(dict(meta=meta, models=model_rows, rows=rows, mt=mt_rows), ensure_ascii=False)
    payload = payload.replace('</', '<\\/')
    return DASHBOARD_TEMPLATE.replace('__DATA__', payload)


dashboard_html = build_dashboard()
DASH_PATH = OUT_DIR / 'dashboard.html'
DASH_PATH.write_text(dashboard_html, encoding='utf-8')
print(f'대시보드 저장: {DASH_PATH}')
display(HTML(dashboard_html))

Output hidden; open in https://colab.research.google.com to view.

## 12. 결과 파일과 읽는 법

| 파일 | 내용 |
|---|---|
| `translations/후보이름.parquet` | 번역 후보별 번역문 (review_id, ko) |
| `mt_compare.csv` | 번역 후보 비교 표 |
| `mt_samples.csv` | 원문, 후보별 번역문, 역번역 예시 |
| `results/모델이름.json` | 한국어 감성 모델별 학습 기록과 테스트 예측 |
| `failed_models.json` | 실패한 감성 모델과 오류 내용 |
| `metrics_summary.csv`, `metrics_by_item.csv`, `metrics_full.json` | KO·EN 모델 지표 |
| `metrics_en_vs_ko.csv` | 같은 모델의 영어 → 한국어 정확도 변화 |
| `predictions_test.csv` | 원래값/예측값 표 (원문, 번역문, 번역 후 오답 표시) |
| `dashboard.html` | 비교 화면 |

**읽을 때 참고**
- 정확도가 영어보다 크게 떨어지면 번역에서 감성이 바뀌었을 가능성이 큽니다. `번역 후 오답` 리뷰를 몇 개 직접 읽어 보세요.
- KcELECTRA·KLUE가 다국어 모델보다 좋다면, 실제 한국 리뷰에서도 한국어 전용 모델이 유리할 가능성이 높습니다. 다만 번역문은 실제 한국 리뷰와 말투가 달라서 최종 판단은 한국 리뷰로 다시 확인해야 합니다.